# Reproducing Vijayan et al. (2022) from dandiset 000212

Paper. Vijayan V, Wang Z, Chandra V, Chakravorty A, Li R, Sarbanes SL, Akhlaghpour H, Maimon G
(2022). An internal expectation guides Drosophila egg-laying decisions. *Science Advances* 8,
eabn3852. [doi:10.1126/sciadv.abn3852](https://doi.org/10.1126/sciadv.abn3852). The claims file
(`claims/vijayan-2022.yaml`) was written from the version of record in Europe PMC (PMC9616500), with
the supplement's figures and legends; Tables S1 and S2 were not available to it.

Dandiset. [DANDI:000212](https://dandiarchive.org/dandiset/000212/draft), version `draft`. The
dandiset has no published version: the archive lists only the draft, so the draft is pinned, and
the notebook prints its modification time (2023-06-20T00:56:15Z when this attempt was made). The
paper cites the dandiset without a version (Data and materials availability).

What the dandiset holds. 1013 NWB files, one per fly, in 33 chamber and genotype groups: at 2
frames per second, the fly's position, its island, the sucrose concentration under it and the
frames at which it laid an egg. They include the paper's tracked chambers (two-, three- and
five-island sucrose chambers, the tunnel chambers, the DL, TUT and D. suzukii strains, DDC>Kir at
four temperatures, three dopamine receptor mutants, and four agarose chambers, whose files also
give the agarose under the fly) and the CS data of ref. 16 that the paper reuses. They do not include
the movable-boundary and glued-fly experiments, the Gal4 screen, the Gal80 rescues, nine of the
receptor mutants, the mutants' 200/500 mM and agarose chambers, four of the six agarose conditions
of Fig. 6B, or the feeding-latency videos.

Method. The shared methods M1 to M8 of the claims file are implemented in `egglay.py`, which
says where each parameter comes from. The paper used MATLAB; its rank sum test is computed with a
tie correction, as MATLAB's `ranksum` does. No analysis uses random draws, so no seed is needed.

| Claim | Tested |
|---|---|
| leave-higher-stay-lower, fig2bc, fig2d, fig2e, fig2efg, fig2-earlier-merge, fig3ab, fig3c, fig3d, fig3e, fig3f, fig4a, fig4b, fig4d, fig4e, fig4f, fig5ab, fig5c, fig5de, fig5f, fig5gh, fig6cd, fig6ef, fig6g, fig7cf, fig7df, fig7gh, fig7ik, fig8ce, fig8fg, figs2, figs3, figs4 | tested |
| fig1d, fig1f, fig6b, fig7a, fig7b, fig7l, fig7mn, fig8a, fig8b, fig8j, fig8k, dop1r1-84714 | blocked: data missing |

Revision after the first verification round. The first version of this notebook read only the
island and sucrose series and blocked the agarose claims for want of a hardness label; the agarose
files hold one (`Agarose_Percentage`), and fig6cd and fig6g are now tested, while fig6b stays
blocked because four of its six conditions are not in the dandiset. The other changes answer the
verifier's points: brief boundary visits (fig3d, fig3e), short inter-egg intervals, the one-frame
resolution of pause times, the merges that rest on near-empty bins, the trial and first-bin readings
of the bin test (fig5ab, fig7cf, figs2), a non-negative kernel fit (figs4), and the criteria of
fig4e, fig4f, fig6ef and fig8fg applied in their cells.

In [ ]:
import time
import warnings
from collections import defaultdict
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import egglay as E

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
info = E.version_info()
assets = E.list_assets()
flies = E.load_all(assets)
print(f"dandiset {E.DANDISET} version {info['version']}, modified {info['modified']}, "
      f"{info['asset_count']} assets, {info['size'] / 1e9:.2f} GB")
print(f"loaded {len(flies)} files in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = pd.DataFrame([{k: f[k] for k in ("path", "group", "description", "genotype", "n", "t0", "dt_min",
                                         "dt_max", "same_t", "egg_sum", "pos_nan")}
                     | {"eggs": len(f["eggs"]), "egg_values": str(f["egg_values"]), "units": str(f["units"])}
                     for f in flies])
print("Stated units (all files):", meta.units.unique().tolist())
print("Frame intervals (s): min", meta.dt_min.min(), "max", meta.dt_max.max(), "; first timestamp", meta.t0.unique().tolist())
print("Egg series shares the position timestamps in every file:", bool(meta.same_t.all()))
print("Egg series values:", meta.egg_values.unique().tolist(), "; sum of egg series equals the count of 1s:",
      bool((meta.egg_sum == meta.eggs).all()))
print("Frames with a missing position:", int(meta.pos_nan.sum()))
by = meta.groupby("group").agg(flies=("path", "size"), eggs=("eggs", "sum"), hours=("n", lambda n: n.mean() / 7200),
                               genotype=("genotype", "first"))
print("\nFlies, eggs and mean recording length (h) per chamber and genotype group:")
print(by.round(1).to_string())
print("\nTotal flies", len(meta), "; total eggs", int(meta.eggs.sum()))

In [ ]:
for f in flies:
    E.prepare(f)
rows = []
for g, fs in sorted(defaultdict(list, {k: [f for f in flies if f["group"] == k] for k in by.index}).items()):
    isl_suc = defaultdict(set)
    for f in fs:
        for i in np.unique(f["island"]):
            isl_suc[int(i)] |= set(np.unique(f["sucrose"][f["island"] == i]).astype(int).tolist())
    xy0 = np.min([f["xy_min"] for f in fs], axis=0); xy1 = np.max([f["xy_max"] for f in fs], axis=0)
    sp = np.concatenate([f["speed"] for f in fs])
    rows.append({"group": g, "islands": len(isl_suc), "sucrose by island (mM)": dict(isl_suc),
                 "x range (mm)": f"{xy0[0]:.1f} to {xy1[0]:.1f}", "y range (mm)": f"{xy0[1]:.1f} to {xy1[1]:.1f}",
                 "speed median": round(float(np.median(sp)), 3), "speed p99": round(float(np.percentile(sp, 99)), 2)})
print(pd.DataFrame(rows).to_string(index=False))
# island order along y, and sucrose changes only at island changes
order_ok = all(all(f["y_island"][i][0] <= f["y_island"][i + 1][1] for i in range(len(f["y_island"]) - 1)) for f in flies)
suc_without_island = sum(int(((np.diff(f["sucrose"]) != 0) & (np.diff(f["island"]) == 0)).sum()) for f in flies)
print("\nIslands are numbered in order of y position in every file:", order_ok)
print("Frames where the sucrose label changes without an island change:", suc_without_island)

The files hold one fly each, at 2 frames per second (every interval is 0.5 s), with position in mm,
the island index, the sucrose concentration under the fly and a 0/1 egg series on the same
timestamps. The chambers are 20 mm wide (two-island chambers about 20 by 20 mm, three-island 20 by
32, five-island 20 by 56, the two-circle tunnel chambers about 30 by 60), consistent with fig. S1.
Island numbers follow the y position, so an island is a place in the chamber; the sucrose label
says which substrate is there, and it changes only where the island changes. Which substrate sits
on which island differs between flies of a group (the chambers were loaded both ways round), so the
substrate is read from the sucrose series, not from the island number.

In [ ]:
# The paper's counts that the files can be checked against (legends of Figs. 3 to 8 and fig. S3).
paper_counts = {"0-200-CS": 1862, "0-200-TUT": 1184, "0-200-DL": 766, "0-200-suzukii": 214,
                "0-200-DDC-18C": 1029, "0-200-DDC-27C": 761, "0-200-DDC-29CAND295C": 540, "0-200-DDC-31C": 533,
                "0-200-Dop1R1-mutant": 498, "0-200-Dop1R2-mutant-1": 586, "0-200-Dop1R2-mutant-2": 323,
                "0-200-CS-twocircles-4mm-tunnel": 979, "0-200-CS-twocircles-2mm-tunnel": 918,
                "0-200-CS-twocircles-2mm-tunnel-bounce": 675, "0-500-200-CS": 2312, "500-0-500-CS": 943,
                "500-0-200-CS": 1946, "0p8--0p8--CS": 831, "1p4--1p4--CS": 795}
cmp = pd.DataFrame([{"group": g, "eggs in files": int(by.loc[g, "eggs"]), "paper (egg-aligned averages)": n,
                     "difference": int(by.loc[g, "eggs"]) - n} for g, n in paper_counts.items()])
print(cmp.to_string(index=False))
ref16 = ["0-200-CS", "0-500-CS", "500-200-CS"]
print("\nfig. S3: 0/200, 0/500 and 200/500 mM chambers:", int(by.loc[ref16, "flies"].sum()), "flies,",
      int(by.loc[ref16, "eggs"].sum()), "eggs (paper: 95 flies, 3979 eggs)")
print("Genotype strings of the Dop1R2 groups:", by.loc[["0-200-Dop1R2-mutant-1", "0-200-Dop1R2-mutant-2"], "genotype"].tolist())

The egg counts per group are within a few eggs of the counts the paper gives for its egg-aligned
averages, which leave out eggs near the start or end of a video, so the groups are the paper's
groups. The 2-mm bounce chamber has 670 eggs against the paper's 675, and D. suzukii 220 against
214. The two Dop1R2 groups are named only by the paper that made the mutant. "Keleman et al Nature
2012" is ref. 44, the source of Dop1R2 attP, and "Deng et al Neuron 2019" is ref. 41, the source of
Dop1R2- (BDSC 84715) (Materials and Methods, Flies); the egg counts (596 and 324 against 586 and
323) agree with that mapping. The Dop1R1 group is "from Handler et al Cell 2019", ref. 43, which is
Dop1R1-.

In [ ]:
# Recordings deposited twice: compare the start of every position trace and every egg-time pattern.
import hashlib
pos_keys = defaultdict(list)
for a, f in zip(assets, flies):
    pos_keys[(f["n"], round(float(f["speed"][:4000].sum()), 3))].append(f["path"])
egg_keys = defaultdict(list)
for f in flies:
    if len(f["eggs"]) > 2:
        egg_keys[hashlib.md5(np.diff(f["eggs"]).tobytes()).hexdigest()].append(f["path"])
print("Pairs with the same length and the same speed trace over the first 2000 s:",
      [v for v in pos_keys.values() if len(v) > 1])
print("Pairs with the same intervals between eggs:", [v for v in egg_keys.values() if len(v) > 1])
iei = np.concatenate([np.diff(f["eggs"]) * E.DT for f in flies if len(f["eggs"]) > 1])
print(f"\nIntervals between successive eggs of a fly: {len(iei)}; under 10 s: {(iei < 10).sum()}, "
      f"under 60 s: {(iei < 60).sum()}; shortest {iei.min():.1f} s")
short = pd.Series([f["group"] for f in flies for d in np.diff(f["eggs"]) * E.DT if d < 10]).value_counts()
print("Intervals under 10 s by group:", short.to_dict())

In [ ]:
# Successive eggs of a fly under 10 s apart; the paper says eggs are at least 1 min apart (Methods,
# Calculation of egg-laying rate). Each pair is listed with what the fly did between the two eggs.
rows = []
for f in flies:
    e = f["eggs"]; d = np.diff(e)
    for k in np.flatnonzero(d * E.DT < 10):
        a, b = e[k], e[k + 1]
        rows.append({"group": f["group"], "fly": int(f["path"].split("/")[0].split("fly#-")[1]), "first egg frame": int(a),
                     "gap (s)": d[k] * E.DT, "same island": bool(f["island"][a] == f["island"][b]),
                     "max speed between (mm/s)": round(float(f["speed"][a:b + 1].max()), 2),
                     "s from second egg to end": (f["n"] - 1 - b) * E.DT})
short_iei = pd.DataFrame(rows)
print(f"{len(short_iei)} pairs of successive eggs under 10 s apart, in {short_iei.group.nunique()} groups; on the same island: "
      f"{int(short_iei['same island'].sum())}; max speed between below 0.3 mm/s: {int((short_iei['max speed between (mm/s)'] < 0.3).sum())}")
print(short_iei.to_string(index=False))
LATE_S = 30
late = [(f["group"], int(f["path"].split("fly#-")[1].split("/")[0]), int((f["eggs"] >= f["n"] - LATE_S * E.FPS).sum()),
         ((f["n"] - 1 - f["eggs"][f["eggs"] >= f["n"] - LATE_S * E.FPS]) * E.DT).tolist()) for f in flies if (f["eggs"] >= f["n"] - LATE_S * E.FPS).any()]
print(f"\nEggs annotated in the last {LATE_S} s of a recording (group, fly, eggs, s before the end):")
for x in late:
    print(" ", x)

In [ ]:
import h5py, io, requests
a0 = assets[0]
buf = io.BytesIO(requests.get(E._s3_url(a0["asset_id"]), timeout=300).content)
from pynwb import NWBHDF5IO
with h5py.File(buf, "r") as h, NWBHDF5IO(file=h, load_namespaces=True) as io_:
    nwb = io_.read()
    print("Example file read with pynwb:", a0["path"])
    print("session_start_time:", nwb.session_start_time, "; session_description:", nwb.session_description)
    print("subject:", nwb.subject.subject_id, "|", nwb.subject.description, "|", nwb.subject.genotype, "|", nwb.subject.sex, nwb.subject.species)
    for name, obj in nwb.processing["behavior"].data_interfaces.items():
        print(" ", name, type(obj).__name__)
    print("Everything in the file:")
    def walk(g, p=""):
        for k, v in g.items():
            if isinstance(v, h5py.Dataset):
                if not p.startswith("specifications"):
                    print("   ", p + k, v.shape, v.dtype, {kk: vv for kk, vv in v.attrs.items() if kk in ("unit", "interval", "conversion")})
            else:
                walk(v, p + k + "/")
    walk(h)

No recording appears twice. Every file carries the same placeholder session start time, so the
files hold no recording dates and the order of experiments cannot be recovered. 48 of 29,531
intervals between eggs are under 10 s (the paper says eggs are separated by at least 1 min, Materials
and Methods, Calculation of egg-laying rate). In 47 of the 48 pairs both eggs are on one island, and
in 34 the fly does not move faster than 0.3 mm/s between them, as when one egg is annotated twice or
two eggs are laid in one pause. Seven pairs come from the ends of two Dop1R2-mutant-1 recordings (flies 11 and
12), whose last 19 s hold 6 and 4 eggs while the fly walks at up to 8.7 and 7.0 mm/s: these 10 eggs
are unlikely to be real. They are left out of the egg-aligned averages, which drop eggs within 2 min
of either end, and the group's egg-aligned count matches the legend's 586; the group holds 596
eggs in all. All eggs are kept as annotated, as the paper says no data were excluded; the fig8fg
section shows the egg-count comparison without them.

In [ ]:
G0 = defaultdict(list)
for f in flies:
    G0[f["group"]].append(f)
# Clock check: the egg series and the position series share one timestamp array (above). A contrast
# that a shared clock predicts: flies search (walk faster) in the half minute before an egg and pause
# at it. Per egg, mean speed over 5 to 30 s before the egg minus mean speed within 1 s of it, at the
# annotated times and at the same times shifted.
def contrast(f, shift):
    e = f["eggs"] + shift
    e = e[(e - 60 >= 0) & (e + 2 < f["n"])]
    s = f["speed"]
    return np.array([s[x - 60:x - 10].mean() - s[x - 2:x + 3].mean() for x in e])
rows = []
for shift_s in (-240, -60, -30, 0, 30, 60, 240):
    v = np.concatenate([contrast(f, int(shift_s * E.FPS)) for f in flies])
    rows.append({"shift (s)": shift_s, "eggs": len(v), "mean contrast (mm/s)": round(float(v.mean()), 3),
                 "fraction of eggs with contrast > 0": round(float((v > 0).mean()), 3)})
print(pd.DataFrame(rows).to_string(index=False))
per = pd.DataFrame([{"group": g, "true": round(float(np.concatenate([contrast(f, 0) for f in G0[g]]).mean()), 2),
                     "shift +60 s": round(float(np.concatenate([contrast(f, 120) for f in G0[g]]).mean()), 2),
                     "shift -60 s": round(float(np.concatenate([contrast(f, -120) for f in G0[g]]).mean()), 2)}
                    for g in by.index if by.loc[g, "eggs"] > 0])
print("\nMean contrast (mm/s) per group:")
print(per.set_index("group").T.to_string())

Flies walk faster in the half minute before an annotated egg than at it (mean contrast 0.83
mm/s, positive for 82% of eggs), while at the same times shifted by 60 s or 4 min the contrast is
near 0 (-0.06 to 0.04 mm/s), and the true contrast is positive in every group (0.35 to 1.50 mm/s)
while the shifted ones lie between -0.16 and 0.17. The egg annotations and the position series
are on one clock, as far as the search before an egg shows it; the sucrose and island series are
derived from position (they change only together, above).

In [ ]:
# Agarose chambers: the four agarose groups carry an Agarose_Percentage series (the agarose under the fly),
# which the other groups do not. Which agarose each island holds, per fly, and whether the series changes only
# with the island:
for g in ["0p8--1p4--CS", "0p8-200-1p4-200-CS", "0p8--0p8--CS", "1p4--1p4--CS"]:
    fs = [f for f in flies if f["group"] == g]
    orient = pd.Series([" / ".join(f"island {i}: {np.unique(f['agarose'][f['island'] == i]).tolist()}" for i in (0, 1)) for f in fs]).value_counts()
    print(f"{g}: {len(fs)} flies; agarose (tenths of a percent) by island: {orient.to_dict()}; frames without a label "
          f"{sum(int((f['agarose'] < 0).sum()) for f in fs)}; label changes without an island change "
          f"{sum(int(((np.diff(f['agarose']) != 0) & (np.diff(f['island']) == 0)).sum()) for f in fs)}")
print("Groups with an agarose series:", sorted({f["group"] for f in flies if f["agarose"] is not None}))
# Per-fly fraction of eggs on island 0, with the agarose of island 0:
for g in ["0p8--1p4--CS", "0p8-200-1p4-200-CS"]:
    fr = [(int(f["agarose"][f["island"] == 0][0]), np.mean(f["island"][f["eggs"]] == 0)) for f in flies if f["group"] == g and len(f["eggs"])]
    for a in (8, 14):
        x = np.array([v for k, v in fr if k == a])
        print(f"{g}, island 0 holds {a / 10}%: {len(x)} flies with eggs; fraction of eggs on island 0, sorted:", np.round(np.sort(x), 2).tolist())
print("Subject descriptions:", {f["group"]: f["description"] for f in flies if f["group"].startswith(("0p8", "1p4"))})

In [ ]:
# Every attribute of every group and dataset in one 0.8/1.4% file (descriptions, comments, units), and
# every short text value, to look for a record of which island held 0.8% agarose.
a08 = next(a for a in assets if E.group_of(a["path"]) == "0p8--1p4--CS")
print(a08["path"])
with h5py.File(io.BytesIO(requests.get(E._s3_url(a08["asset_id"]), timeout=300).content), "r") as h:
    def show(name, obj):
        if name.startswith("specifications"):
            return
        at = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in obj.attrs.items() if k not in ("namespace", "object_id")}
        if isinstance(obj, h5py.Dataset) and obj.dtype.kind in "SOU" and obj.size == 1:
            v = obj[()]
            at["value"] = v.decode() if isinstance(v, bytes) else v
        print(" ", name, at)
    h.visititems(show)

The four agarose groups carry an `Agarose_Percentage` series, which says which agarose is under the
fly in every frame (the first round of this attempt read only the island and sucrose series and
missed it). Its label changes only with the island. The chambers were loaded both ways round: in the
0.8% against 1.4% chamber island 0 is 0.8% for 18 flies and 1.4% for 6, and in the chamber with 200 mM
on both sides 12 and 12. The per-fly egg fractions on island 0 are bimodal for this reason: the
flies lay most eggs on whichever island holds 0.8%. The claims on the softer and harder sides (Fig.
6B, 6C, 6D and 6G) are tested with this series.

In [ ]:
# What the dandiset holds against the experiments the claims need.
need = {
    "Fig. 1D, 1E: movable-boundary chambers": None, "Fig. 1F: glued tarsi or proboscis": None,
    "Fig. 6B bars 2, 3, 6 and one of 4, 5: sucrose on one agarose side, 500 on 0.8% against 200 on 1.4%": None,
    "Fig. 7A, fig. S5: 115 Gal4 lines of the screen": None, "Figs. 7B, 8B: latency to feeding (15 frames/s video)": None,
    "Fig. 7L to N: Gal80 rescues (TH, R58E02, TSH, OTD-FLP)": None,
    "Fig. 8A: DopEcR, Dop2R, 5HT receptor mutants, Dop1R1 f02676": None,
    "Fig. 8J: 200 against 500 mM chambers for mutants and DDC>Kir": None, "Fig. 8K: 0.8 against 1.4% for mutants": None,
    "Methods: Dop1R1 BDSC 84714": None}
print("Groups in the dandiset:", sorted(by.index))
print("\nExperiments the claims need that match no group (by chamber and genotype in the subject description):")
for k in need:
    print("  ", k)
print("\nGroups in the dandiset that no claim uses: 0-0-CS, 200-200-CS, 500-500-CS (Fig. 1C, ref. 16),"
      " 0-200-VT040574-kir and -kirmutant (ref. 16, oviDN silencing)")

In [ ]:
G = defaultdict(list)
for f in flies:
    G[f["group"]].append(f)
_cache = {}


def R(groups, cur, prev, key="sucrose", mode="working"):
    """Rate function M4 for current substrate `cur` since leaving `prev`, pooled over the groups."""
    groups = (groups,) if isinstance(groups, str) else tuple(groups)
    k = (groups, cur, prev, key, mode)
    if k not in _cache:
        fl = [f for g in groups for f in G[g]]
        _cache[k] = E.rate_events(E.labelled(fl, key), cur, prev, mode=mode)
    return _cache[k]


def R_entry(groups, cur=0, mode="working"):
    """Rate on `cur` by time since entering it (since leaving any other substrate)."""
    groups = (groups,) if isinstance(groups, str) else tuple(groups)
    k = (groups, cur, "entry", mode)
    if k not in _cache:
        fl = [dict(label=E.entry_label(f, cur), eggs=f["eggs"], starts=f["starts"], n=f["n"]) for g in groups for f in G[g]]
        _cache[k] = E.rate_events(fl, cur, -999, mode=mode)
    return _cache[k]


EDGES = E.RATE_EDGES
BINS = [f"{int(a)}-{int(b)}" for a, b in zip(EDGES[:-1], EDGES[1:])]


def curve_table(named, pairs=None, reading="visit"):
    """Rates per bin of several curves (eggs/min), with eggs and visits, and the bin p-values of given pairs."""
    df = pd.DataFrame({"bin (s)": BINS})
    for name, r in named.items():
        df[name] = np.round(r["rate"], 3)
        df[name + " eggs/visits"] = [f"{e}/{v}" for e, v in zip(r["eggs"], r["visits"])]
    for a, b in (pairs or []):
        df[f"p {a} vs {b}"] = np.round(E.bin_pvals(named[a], named[b], reading), 4)
    return df


def merge(r1, r2, reading="visit", first_left=5):
    """Merge point M5: the first bin with left edge >= first_left whose p > 0.05 (None: no merge)."""
    p = E.bin_pvals(r1, r2, reading)
    for i in range(len(EDGES) - 1):
        if EDGES[i] >= first_left and np.isfinite(p[i]) and p[i] > 0.05:
            return float(EDGES[i])
    return None


def bins_in(lo, hi):
    return [i for i in range(len(EDGES) - 1) if EDGES[i] >= lo and EDGES[i + 1] <= hi]


def plot_curves(ax, named, title):
    x = EDGES[:-1]
    for name, r in named.items():
        lo, hi = E.rate_ci(r)
        ax.step(np.arange(len(x)), r["rate"], where="post", label=name)
        ax.fill_between(np.arange(len(x)), lo, hi, step="post", alpha=0.2)
    ax.set_xticks(np.arange(len(x))); ax.set_xticklabels([int(v) for v in x], rotation=90, fontsize=7)
    ax.set_title(title, fontsize=8); ax.set_ylabel("eggs/min"); ax.legend(fontsize=6)
    top = max(np.nanmax(r["rate"]) for r in named.values())
    ax.set_ylim(0, max(top * 1.1, 0.1))

In [ ]:
# Size of what rate_events counts twice: frames of a working search window (30-s minimum) that lie in an
# earlier window of the same fly as well.
n_tot = n_rep = 0
for f in flies:
    if len(f["eggs"]) == 0:
        continue
    s0, e0 = E.search_windows(f["starts"], f["eggs"], f["n"], "working")
    cnt = np.zeros(f["n"], np.int16)
    for s, e in zip(s0, e0):
        cnt[s:e + 1] += 1
    n_tot += int(cnt.sum()); n_rep += int((cnt[cnt > 1] - 1).sum())
print(f"Working search windows hold {n_tot} frame counts; {n_rep} of them ({100 * n_rep / n_tot:.1f}%) repeat a frame of an earlier window of the same fly")

## leave-higher-stay-lower

> In two-choice sucrose chambers, flies leave the higher sucrose substrate in 964 of 1205 (80%)
> searches that start on it, and leave the lower sucrose substrate in only 918 of 2744 (33%)
> searches that start on it.

Searches by M3 without the 30-s minimum, in the CS 0/200, 0/500 and 200/500 mM chambers (95
flies); the substrate at the first frame of the search, and whether the fly is on the other
substrate at any frame before the egg.

In [ ]:
def leave_table(half=E.HALF_BOX):
    rows = []
    for g, hi in [("0-200-CS", 200), ("0-500-CS", 500), ("500-200-CS", 500)]:
        for f in G[g]:
            st = E.search_starts(f["speed"], f["eggs"], half)
            for a, e in zip(st, f["eggs"]):
                seg = f["sucrose"][a:e + 1]
                rows.append({"chamber": g, "fly": f["path"], "start_high": seg[0] == hi, "left": bool((seg != seg[0]).any()),
                             "duration": (e - a) * E.DT})
    return pd.DataFrame(rows)


lv = leave_table()
H = lv[lv.start_high].left.values.astype(int); L = lv[~lv.start_high].left.values.astype(int)
p_leave = E.ranksum01(H, L)
print(f"Searches started on higher sucrose: {len(H)}, left in {H.sum()} ({H.mean():.3f})")
print(f"Searches started on lower sucrose:  {len(L)}, left in {L.sum()} ({L.mean():.3f})")
print(f"Two-sided Wilcoxon rank sum on 0/1 trials: p = {p_leave:.3g}")
print("\nBy chamber:")
print(lv.groupby(["chamber", "start_high"]).left.agg(["size", "sum", "mean"]).round(3).to_string())
pf = lv.groupby(["fly", "start_high"]).left.mean().unstack()
print(f"\nPer fly (flies with searches of both kinds, {pf.dropna().shape[0]}): median fraction leaving higher "
      f"{pf[True].median():.2f}, lower {pf[False].median():.2f}; flies where higher > lower: {(pf[True] > pf[False]).sum()}")
v = leave_table(half=1)
print(f"\nVariant, search scan starting at the frame before the egg: higher {v.start_high.sum()} searches, "
      f"{v[v.start_high].left.mean():.3f} leave; lower {(~v.start_high).sum()}, {v[~v.start_high].left.mean():.3f} leave")
print("Search durations (s), M3 without the 30-s minimum: min", lv.duration.min(), "median", lv.duration.median(), "max", lv.duration.max())

**Result: reproduced.** 1205 searches start on the higher sucrose substrate and 80.5% of them
leave it (970); 2773 start on the lower and 33.1% leave it (917); p = 8.8e-167 by the rank sum
test on 0/1 trials. All four values are within tolerance, and the direction holds in each of the
three chambers and in 87 of the 88 flies with searches of both kinds.

## fig2bc-larger-chambers-two-options

> In three- and five-island chambers with two sucrose concentrations, flies choose the low sucrose
> option with egg-laying rate dynamics similar to standard two-choice chambers.

Fig. 2B is the 0/200/200 chamber; the dandiset's only five-island chamber with two concentrations
is 0/200/200/200/200, taken as Fig. 2C.

In [ ]:
sel = bins_in(5, 60)
res_2bc = {}
fig, axs = plt.subplots(1, 3, figsize=(13, 3))
for ax, g in zip(axs, ["0-200-CS", "0-200-200-CS", "0-200-200-200-200-CS"]):
    r0, r2 = R(g, 0, 200), R(g, 200, 0)
    p = E.bin_pvals(r0, r2)
    below = all(r2["rate"][i] < r0["rate"][i] for i in sel)
    nsig = sum(p[i] < 0.05 and r2["rate"][i] < r0["rate"][i] for i in sel)
    res_2bc[g] = below and nsig >= 1
    eg = [f["eggs"] for f in G[g]]
    on0 = sum(int((f["sucrose"][f["eggs"]] == 0).sum()) for f in G[g])
    print(f"{g}: eggs {sum(map(len, eg))}, on 0 mM {on0} ({on0 / sum(map(len, eg)):.3f}); 200 below 0 in all bins 5-60 s: {below}; "
          f"bins 5-60 s with p < 0.05: {nsig} of {len(sel)}; merge point (visit reading) {merge(r0, r2)}, (search reading) {merge(r0, r2, 'search')}")
    print(curve_table({"0 since 200": r0, "200 since 0": r2}, [("0 since 200", "200 since 0")]).iloc[1:9].to_string(index=False))
    plot_curves(ax, {"0 since 200": r0, "200 since 0": r2}, g)
plt.tight_layout(); plt.show()
print("Criterion holds in the 0/200/200 chamber:", res_2bc["0-200-200-CS"], "; in the five-island chamber:", res_2bc["0-200-200-200-200-CS"])

**Result: reproduced.** In the 0/200/200 and the five-island chamber the rate on 200 mM since
leaving 0 mM is below the rate on 0 mM since leaving 200 mM in all 7 bins from 5 to 60 s, all 7 at
p < 0.05, as in the two-choice 0/200 chamber; 80.3% and 82.0% of eggs are on 0 mM. In none of the
three chambers do the curves merge within 7200 s.

## fig2d-three-choice-inhibit-200

> In a 0/500/200 mM chamber flies inhibit egg laying upon entering 200 mM from 500 mM, although
> 200 mM is strongly preferred after the same transition in two-choice chambers.

In [ ]:
r3, r2c = R("0-500-200-CS", 200, 500), R("500-200-CS", 200, 500)
for reading in ("visit", "search"):
    p = E.bin_pvals(r3, r2c, reading)
    lower = sum(r3["rate"][i] < r2c["rate"][i] for i in sel)
    sig = sum(p[i] < 0.05 and r3["rate"][i] < r2c["rate"][i] for i in sel)
    print(f"{reading} trials: three-choice lower in {lower} of {len(sel)} bins from 5 to 60 s; lower at p < 0.05 in {sig}")
print(curve_table({"200 since 500, 0/500/200": r3, "200 since 500, 200/500": r2c},
                  [("200 since 500, 0/500/200", "200 since 500, 200/500")]).iloc[:10].to_string(index=False))
e3 = pooled = E.pooled_trials(r3, 5, 60); e2 = E.pooled_trials(r2c, 5, 60)
print(f"Pooled 5 to 60 s: 0/500/200 {e3[0]:.3f} eggs/min ({e3[2]} eggs), 200/500 {e2[0]:.3f} ({e2[2]} eggs)")

**Result: reproduced.** On entering 200 mM from 500 mM, the 0/500/200 rate is below the 200/500
rate in all 7 bins from 5 to 60 s, all 7 at p < 0.05 under both trial readings (pooled over the
window, 0.239 against 1.093 eggs/min).

## fig2e-200-recovers-with-time

> In the 0/500/200 mM chamber, the egg-laying rate on 200 mM rises gradually with time since the
> last visit to 0 mM.

In [ ]:
r = R("0-500-200-CS", 200, 0)
late = E.pooled_trials(r, 300, 7200); early = E.pooled_trials(r, 5, 30)
for reading in ("visit", "search"):
    lt = E.pooled_trials(r, 300, 7200, reading)[1]; et = E.pooled_trials(r, 5, 30, reading)[1]
    print(f"{reading} trials: late {lt.sum()}/{len(lt)}, early {et.sum()}/{len(et)}, p = {E.ranksum01(lt, et):.3g}")
print(f"Rate on 200 mM since 0 mM: bins from 300 s {late[0]:.3f} eggs/min ({late[2]} eggs, {late[3]} frames); bins 5 to 30 s {early[0]:.3f} ({early[2]} eggs)")
rr = r["rate"][1:]
steps = np.diff(rr)
print("Rates from the 5-10 s bin on:", np.round(rr, 3).tolist())
print(f"Successive bins non-decreasing: {(steps >= 0).sum()} of {len(steps)}")
fig, ax = plt.subplots(figsize=(5, 3)); plot_curves(ax, {"200 since 0": r, "0 since 200": R("0-500-200-CS", 0, 200)}, "0/500/200"); plt.show()

**Result: reproduced.** The rate on 200 mM is 1.478 eggs/min in the bins from 300 s since leaving 0
mM (233 eggs) against 0.088 in the bins from 5 to 30 s (52 eggs), p printed as 0 on either trial
reading, and it does not decrease in 11 of 12 successive bins from 5 s on.

## fig2efg-inhibit-200-after-0

> In all three-choice chambers flies inhibit egg laying on 200 mM nearly completely if they visited
> 0 mM in the last ~30 s, and this holds with a larger distance between 0 and 200 mM
> (0/500/500/500/200) and with several 500 to 200 mM transitions along the path (0/500/200/500/200).

In [ ]:
three = ["0-500-200-CS", "0-500-500-500-200-CS", "0-500-200-500-200-CS"]
rows = []
for lo in (5, 10):
    for g in three:
        a = E.pooled_trials(R(g, 200, 0), lo, 30); b = E.pooled_trials(R(g, 0, 200), lo, 30)
        p = E.ranksum01(a[1], b[1])
        ps = E.ranksum01(E.pooled_trials(R(g, 200, 0), lo, 30, "search")[1], E.pooled_trials(R(g, 0, 200), lo, 30, "search")[1])
        rows.append({"window": f"{lo}-30 s", "chamber": g, "rate 200 since 0": round(a[0], 4), "eggs 200": a[2], "frames 200": a[3],
                     "rate 0 since 200": round(b[0], 4), "eggs 0": b[2], "ratio": round(a[0] / b[0], 4),
                     "p visit": p, "p search": ps, "<=10%": a[0] / b[0] <= 0.1, "<=20%": a[0] / b[0] <= 0.2})
t2efg = pd.DataFrame(rows)
print(t2efg.to_string(index=False))
main = t2efg[t2efg.window == "5-30 s"]
ok10 = int((main["<=10%"] & (main["p visit"] < 0.05)).sum()); ok20 = int((main["<=20%"] & (main["p visit"] < 0.05)).sum())
print(f"\nWindow 5-30 s: chambers meeting reading (a) 10% with p < 0.05: {ok10} of 3; reading (b) 20%: {ok20} of 3")

**Result: reproduced.** Over 5 to 30 s since leaving 0 mM, the rate on 200 mM is 14.7%, 9.7% and
7.6% of the rate on 0 mM since leaving 200 mM in the 0/500/200, 0/500/500/500/200 and
0/500/200/500/200 chambers, each difference at p < 1e-06. Reading (b), at most 20%, holds in all
three; reading (a), at most 10%, holds in two (not 0/500/200). The 10 to 30 s window gives the same
(15.3%, 9.8%, 6.7%).

## fig2-earlier-merge-three-choice

> The 0 and 200 mM rate functions merge at an earlier time in three-choice chambers than in
> two-choice chambers.

In [ ]:
INF = float("inf")
def mp(g, reading="visit", first_left=5):
    m = merge(R(g, 0, 200), R(g, 200, 0), reading, first_left)
    return INF if m is None else m
rows = []
for first_left in (5, 10):
    for reading in ("visit", "search"):
        mt = {g: mp(g, reading, first_left) for g in three}
        ca = {"0-200-CS": mp("0-200-CS", reading, first_left)}
        cb = ca | {g: mp(g, reading, first_left) for g in ["0-200-200-CS", "0-200-200-200-200-CS"]}
        na = sum(all(mt[g] < v for v in ca.values()) for g in three)
        nb = sum(all(mt[g] < v for v in cb.values()) for g in three)
        rows.append({"first bin": f">= {first_left} s", "trials": reading, **{k.replace("-CS", ""): v for k, v in (mt | cb).items()},
                     "earlier than (a)": na, "earlier than (b)": nb})
print(pd.DataFrame(rows).to_string(index=False))
print("(inf: no bin merges within 7200 s)")
for g in three:
    print(g, "bin p-values, 0 since 200 against 200 since 0:", np.round(E.bin_pvals(R(g, 0, 200), R(g, 200, 0)), 3).tolist())
print("\nEggs/visits per bin from 5 to 60 s, 0 since 200 | 200 since 0 (visit trials), to show what a merge rests on:")
for g in three:
    r0_, r2_ = R(g, 0, 200), R(g, 200, 0); p_ = E.bin_pvals(r0_, r2_)
    print(" ", g, "; ".join(f"{BINS[i]}: {r0_['eggs'][i]}/{r0_['visits'][i]} | {r2_['eggs'][i]}/{r2_['visits'][i]} p={p_[i]:.3f}" for i in bins_in(5, 60)))

**Result: reproduced.** With the first bin taken as 5 to 10 s and visit trials, the 0 and 200 mM
curves merge at 1200, 5 and 5 s in the 0/500/200, 0/500/500/500/200 and 0/500/200/500/200
chambers, while none of the two-choice 0/200 chamber, the 0/200/200 chamber or the five-island
0/200 chamber merges within 7200 s. All three three-choice chambers merge earlier than every
comparison chamber under both readings (a) and (b); with search trials, or with the first bin
taken as 10 to 15 s, at least two of three still do. The support is thinner than the merge points
suggest: the two merges at 5 s rest on bins with almost no eggs (0 of 42 against 0 of 48 visits in
0/500/500/500/200, 4 of 854 against 0 of 715 in 0/500/200/500/200), and the bins after them differ
(p = 0.011 or less from 15 s on in the first, p < 0.001 from 10 s on in the second). The merge at 10
s with the 10 to 15 s bin first is of the same kind (1 of 189 against 1 of 158). Only the 1200 s
merge of 0/500/200 follows bins with many eggs on both curves. The criterion takes the merge point
as the first bin with P > 0.05, so the outcome stands as tested.

## fig3ab-2mm-tunnel-faster

> With a 2-mm tunnel between 200 mM and 0 mM, flies lay eggs at a higher rate in the 5 to 10 s after
> entering 0 mM than with a 4-mm tunnel.

In [ ]:
T4, T2, TB = "0-200-CS-twocircles-4mm-tunnel", "0-200-CS-twocircles-2mm-tunnel", "0-200-CS-twocircles-2mm-tunnel-bounce"


def bin_compare(ga, gb, i, mode="working", reading="visit"):
    ra, rb = R(ga, 0, 200, mode=mode), R(gb, 0, 200, mode=mode)
    key = "trials_visit" if reading == "visit" else "trials_search"
    return ra["rate"][i], rb["rate"][i], E.ranksum01(ra[key][i], rb[key][i]), (ra["eggs"][i], ra["visits"][i]), (rb["eggs"][i], rb["visits"][i])


def eval_3ab(mode="working", reading="visit"):
    a, b, p, ca, cb = bin_compare(T2, T4, 1, mode, reading)
    return (a > b and p < 0.05), (a, b, p, ca, cb)


for reading in ("visit", "search"):
    ok, (a, b, p, ca, cb) = eval_3ab(reading=reading)
    print(f"{reading} trials: 5-10 s bin, 2 mm {a:.3f} eggs/min (eggs/visits {ca}), 4 mm {b:.3f} ({cb}), p = {p:.3g}; holds: {ok}")
print(curve_table({"4 mm": R(T4, 0, 200), "2 mm": R(T2, 0, 200), "bounce": R(TB, 0, 200)}, [("2 mm", "4 mm"), ("bounce", "4 mm"), ("bounce", "2 mm")]).iloc[:6].to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3)); plot_curves(ax, {"4 mm": R(T4, 0, 200), "2 mm": R(T2, 0, 200), "bounce": R(TB, 0, 200)}, "0 mM since leaving 200 mM"); plt.show()

**Result: reproduced.** In the 5 to 10 s bin after entering 0 mM, the rate is 1.236 eggs/min with
the 2-mm tunnel (36 eggs in 369 visits) against 0.364 with the 4-mm tunnel (26 in 875), p = 5.2e-07
on visit trials and 2.2e-04 on search trials.

## fig3c-bounce-faster

> With angled walls that bounce flies away from the 2-mm tunnel, flies lay eggs even more rapidly
> on entering 0 mM than in the 2-mm and 4-mm tunnel chambers.

In [ ]:
def eval_3c(mode="working", reading="visit", i=1):
    a4 = bin_compare(TB, T4, i, mode, reading); a2 = bin_compare(TB, T2, i, mode, reading)
    hi4 = a4[0] > a4[1] and a4[2] < 0.05; hi2 = a2[0] > a2[1] and a2[2] < 0.05
    return ("reproduced" if hi4 and hi2 else "partially_reproduced" if hi4 else "not_reproduced"), (a4, a2)


for i, name in ((1, "(a) 5-10 s"), (0, "(b) 0-5 s")):
    for reading in ("visit", "search"):
        out, (a4, a2) = eval_3c(reading=reading, i=i)
        print(f"{name}, {reading} trials: bounce {a4[0]:.3f} eggs/min {a4[3]}; 4 mm {a4[1]:.3f} {a4[4]} p = {a4[2]:.3g}; "
              f"2 mm {a2[1]:.3f} {a2[4]} p = {a2[2]:.3g} -> {out}")
fisher = stats.fisher_exact([[bin_compare(TB, T2, 1)[3][0], bin_compare(TB, T2, 1)[3][1] - bin_compare(TB, T2, 1)[3][0]],
                             [bin_compare(TB, T2, 1)[4][0], bin_compare(TB, T2, 1)[4][1] - bin_compare(TB, T2, 1)[4][0]]])
print(f"Fisher's exact test, bounce against 2 mm in the 5-10 s bin (visits): p = {fisher.pvalue:.3g} (paper: 0.058 for the second P of Fig. 3C)")

**Result: partially reproduced.** Under reading (a), the 5 to 10 s bin, the bounce chamber's rate
is 2.237 eggs/min (17 eggs in 102 visits), higher than the 4-mm chamber (0.364, p = 1.8e-10) and
higher than the 2-mm chamber (1.236) at p = 0.051, just short of 0.05 (search trials, p = 0.50).
Fisher's exact test on the second comparison gives 0.075, where the paper reports 0.058. Under
reading (b), the 0 to 5 s bin, the bounce chamber has no eggs in 130 visits and is not higher than
either.

## fig3d-dwell-before-fast-eggs

> Across the 4-mm, 2-mm and 2-mm bounce chambers, flies spend progressively more time on 200 mM
> before entering 0 mM and laying a fast egg (within 10 s), with medians of 54, 96 and 576 s over 36,
> 44 and 23 eggs.

In [ ]:
FE = {}
rows = []
for g in (T4, T2, TB):
    fe = [dict(x, fly=f["path"]) for f in G[g] for x in E.fast_eggs(f)]
    fe_lt = [x for x in fe if x["t_in"] < 10]
    FE[g] = fe
    d = np.array([x["dwell"] for x in fe])
    rows.append({"chamber": g, "fast eggs (<= 10 s)": len(fe), "median dwell (s)": np.median(d),
                 "fast eggs (< 10 s)": len(fe_lt), "median (< 10 s)": np.median([x["dwell"] for x in fe_lt]),
                 "flies": len({x["fly"] for x in fe}), "dwell range (s)": f"{d.min():.1f} to {d.max():.1f}"})
print(pd.DataFrame(rows).to_string(index=False))
short = {T4: "4 mm", T2: "2 mm", TB: "bounce"}
for a, b in [(T4, T2), (T2, TB), (T4, TB)]:
    print(f"rank sum, dwell {short[a]} against {short[b]}: p = {E.ranksum([x['dwell'] for x in FE[a]], [x['dwell'] for x in FE[b]]):.3g}")
print("Per fly, fast eggs and median dwell:")
for g in (T4, T2, TB):
    s = pd.DataFrame(FE[g]).groupby("fly").dwell.agg(["size", "median"])
    print(" ", g, "flies", len(s), "; eggs per fly", s["size"].tolist())
# Boundary flicker: the length of every sucrose visit in the tunnel chambers, and the result with brief
# visits merged into the visits around them (a variant; the paper's M1 assigns every frame by the midline).
print("\nFrames with no sucrose label in the tunnel chambers:", sum(int((f["sucrose"] < 0).sum()) for g in (T4, T2, TB) for f in G[g]),
      "(so the first frame on 0 mM is the frame after the last on 200 mM: 'from crossing the midline' and 'from leaving 200 mM' are one)")
for g in (T4, T2, TB):
    d = np.concatenate([(E.visits(f["sucrose"])[1] - E.visits(f["sucrose"])[0] + 1) * E.DT for f in G[g]])
    print(f"  {short[g]}: {len(d)} sucrose visits; at most 0.5 s: {np.sum(d <= 0.5)}, 1 s: {np.sum(d <= 1)}, 2 s: {np.sum(d <= 2)}, "
          f"5 s: {np.sum(d <= 5)}; median {np.median(d):.1f} s; range {d.min()} to {d.max()} s")
print("Variant, visits of at most m s between two visits to one substrate merged into them (fast eggs, median dwell):")
FE_M = {}
for ms in (0.5, 1, 2, 5):
    out = []
    for g in (T4, T2, TB):
        d = [x["dwell"] for f in G[g] for x in E.fast_eggs(dict(f, sucrose=E.merge_short_visits(f["sucrose"], ms)))]
        FE_M[(ms, g)] = d
        out.append(f"{short[g]} {len(d)}, {np.median(d):.1f} s")
    print(f"  m = {ms} s:", "; ".join(out))
print("Tolerances: medians 48.6-59.4, 86.4-105.6, 518.4-633.6 s; counts 32.4-39.6, 39.6-48.4, 20.7-25.3")

**Result: partially reproduced.** The counts of fast eggs, 36, 44 and 22, are within tolerance of 36,
44 and 23, and the median time on 200 mM before a fast egg increases from the 4-mm to the 2-mm to
the bounce chamber, but the medians, 45.5, 57.0 and 379.5 s, are all below the tolerances around
54, 96 and 576 s. The bounce chamber differs from both others (p = 0.003); the 4-mm and 2-mm
chambers do not differ (p = 0.54). No frame of the tunnel chambers lacks a sucrose label, so the
first frame on 0 mM follows the last on 200 mM and the claim's two readings of "within 10 s" (from
crossing the midline, from leaving 200 mM) are one; counting eggs strictly under 10 s gives 34, 42
and 18 with the same medians. Brief visits are common at the boundary (359 of 12154 visits in the
4-mm chamber last one frame), but merging visits of up to 0.5, 1, 2 or 5 s into the visits around
them moves the medians only to 48.5, 57.0 and 441.5 s (448.2 s at 5 s), still outside every
tolerance, with 35, 44 and 21 (or 20) fast eggs. Flicker at the boundary does not explain the short
medians.

## fig3e-rate-after-long-absence

> Pooling the three tunnel chambers, the egg-laying rate in the 10 s after entering 0 mM rises when
> flies have not experienced 0 mM for more than about 4 min.

In [ ]:
E3_EDGES = np.array([0, 5, 10, 15, 20, 25, 30, 45, 60, 90, 150, 240, np.inf])


def trials_3e(merge_s=0):
    """One trial per entry to 0 mM from 200 mM: its first 10 s on 0 mM inside a search, and the length of the 200 mM visit before."""
    trials = []
    for g in (T4, T2, TB):
        for f in G[g]:
            lab = E.merge_short_visits(f["sucrose"], merge_s) if merge_s else f["sucrose"]
            insearch = E.in_search_mask(f, "working")
            st, en, val = E.visits(lab)
            eggset = set(f["eggs"].tolist())
            for i in range(1, len(st)):
                if val[i] != 0 or val[i - 1] != 200:
                    continue
                fr = np.arange(st[i], min(en[i], st[i] + int(10 * E.FPS) - 1) + 1)
                fr = fr[insearch[fr]]
                if len(fr) == 0:
                    continue
                egg = int(any(x in eggset for x in fr))
                trials.append({"chamber": g, "prev200": (en[i - 1] - st[i - 1] + 1) * E.DT, "frames": len(fr), "egg": egg})
    tr = pd.DataFrame(trials)
    tr["bin"] = pd.cut(tr.prev200, E3_EDGES, right=False)
    return tr


def table_3e(tr):
    tb = tr.groupby("bin", observed=False).agg(visits=("egg", "size"), eggs=("egg", "sum"), frames=("frames", "sum"))
    tb["rate (eggs/min)"] = (tb.eggs / tb.frames * 120).round(3)
    return tb


tr = trials_3e()
tb = table_3e(tr)
print(tb.to_string())
long_, short_ = tr[tr.prev200 >= 240], tr[tr.prev200 < 240]
rl = long_.egg.sum() / long_.frames.sum() * 120; rs = short_.egg.sum() / short_.frames.sum() * 120
p3e = E.ranksum01(long_.egg, short_.egg)
lowbins = tb.iloc[:-1]
rho = stats.spearmanr(np.arange(len(lowbins)), lowbins["rate (eggs/min)"])
print(f"\nPrevious 200 mM visit >= 240 s: {rl:.3f} eggs/min ({long_.egg.sum()} eggs / {len(long_)} visits); < 240 s: {rs:.3f} "
      f"({short_.egg.sum()} / {len(short_)}); p = {p3e:.3g}")
print(f"Bins below 240 s: Spearman rho of rate against bin order = {rho.statistic:.3f}, p = {rho.pvalue:.3g}")
print("By chamber, rate for >= 240 s and < 240 s:")
for g in (T4, T2, TB):
    a = tr[(tr.chamber == g) & (tr.prev200 >= 240)]; b = tr[(tr.chamber == g) & (tr.prev200 < 240)]
    print(f"  {g}: {a.egg.sum() / a.frames.sum() * 120:.3f} ({a.egg.sum()}/{len(a)}) against {b.egg.sum() / b.frames.sum() * 120:.3f} ({b.egg.sum()}/{len(b)})")

# Readings of 'the bins below 240 s show no rising trend', with and without brief visits merged (fig3d).
rows = []
for merge_s in (0, 0.5, 1, 2):
    t_ = trials_3e(merge_s); b_ = table_3e(t_)["rate (eggs/min)"].values
    L_, S_ = t_[t_.prev200 >= 240], t_[t_.prev200 < 240]
    mid_, low_ = t_[(t_.prev200 >= 150) & (t_.prev200 < 240)], t_[t_.prev200 < 150]
    r_all = stats.spearmanr(np.arange(11), b_[:11]); r_5 = stats.spearmanr(np.arange(10), b_[1:11]); r_150 = stats.spearmanr(np.arange(9), b_[1:10])
    rows.append({"merged up to (s)": merge_s, ">=240 vs <240": f"{L_.egg.sum() / L_.frames.sum() * 120:.3f} vs {S_.egg.sum() / S_.frames.sum() * 120:.3f}",
                 "p": E.ranksum01(L_.egg, S_.egg), "0-5 s bin eggs/visits": f"{int(t_[t_.prev200 < 5].egg.sum())}/{int((t_.prev200 < 5).sum())}",
                 "rho, bins 0-240": round(r_all.statistic, 2), "p 0-240": round(r_all.pvalue, 4),
                 "rho, bins 5-240": round(r_5.statistic, 2), "p 5-240": round(r_5.pvalue, 4),
                 "rho, bins 5-150": round(r_150.statistic, 2), "p 5-150": round(r_150.pvalue, 4),
                 "150-240 vs <150": f"{mid_.egg.sum() / mid_.frames.sum() * 120:.3f} vs {low_.egg.sum() / low_.frames.sum() * 120:.3f}",
                 "p 150-240 vs <150": E.ranksum01(mid_.egg, low_.egg)})
print("\n" + pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** In the first 10 s on 0 mM, the rate is 1.213 eggs/min after a
previous visit to 200 mM of at least 240 s (25 eggs in 140 visits) against 0.345 after shorter
visits (69 in 1280), p = 1.8e-08, and it is higher in each chamber. The second part, no rising trend
below 240 s, holds only under one reading. Over all bins below 240 s the rank correlation of rate
with bin order is 0.42 (p = 0.20). But the 150 to 240 s bin is already high (0.922 against 0.316
eggs/min below 150 s, p = 0.0014), so over the bins from 5 s the correlation is 0.68 (p = 0.029);
and with brief visits of up to 0.5 s merged (the boundary flicker of fig3d) the 0 to 5 s bin loses
its 2 eggs and the correlation over all bins below 240 s is 0.74 (p = 0.009). Below 150 s the rates
are flat under every variant (p = 0.11 to 0.21). The rise is present but starts between 150 and 240
s, earlier than the claim's 240 s.

## fig3f-motor-program-not-faster

> The delay between the dip in locomotor speed and egg deposition is similar for fast eggs and for
> all eggs in the three tunnel chambers, so the decision and not the deposition motor program is
> faster.

In [ ]:
def aligned_count(groups, window=E.ALIGN_S):
    return sum(len(E.egg_aligned(f, before=window, after=window)) for g in groups for f in G[g])


pz = {g: np.concatenate([f["pause"] for f in G[g]]) for g in (T4, T2, TB)}
pf_ = {}
for g in (T4, T2, TB):
    byfly = {f["path"]: f for f in G[g]}
    pf_[g] = np.array([byfly[x["fly"]]["pause"][list(byfly[x["fly"]]["eggs"]).index(x["egg"])] for x in FE[g]])
pooled_fast = np.concatenate(list(pf_.values()))
rows = []
for g in (T4, T2, TB):
    a = pz[g][np.isfinite(pz[g])]
    for reading, fast in (("(a) own chamber", pf_[g]), ("(b) pooled fast", pooled_fast)):
        rows.append({"chamber": g, "reading": reading, "fast eggs": len(fast), "median pause fast (s)": np.median(fast),
                     "all eggs": len(a), "median pause all (s)": np.median(a),
                     "diff %": round(100 * (np.median(fast) - np.median(a)) / np.median(a), 1), "p": E.ranksum(fast, a)})
t3f = pd.DataFrame(rows)
print(t3f.to_string(index=False))
for w in (60, 120, 300):
    print(f"Eggs in the egg-aligned average with a +-{w} s window:", [aligned_count([g], w) for g in (T4, T2, TB)], "(paper 979, 918, 675)")
fig, ax = plt.subplots(figsize=(6, 3))
tt = np.arange(-E.ALIGN_S * 2, E.ALIGN_S * 2 + 1) * E.DT
for g in (T4, T2, TB):
    m = np.concatenate([E.egg_aligned(f) for f in G[g]]).mean(0); ax.plot(tt, m, label=g.split("twocircles-")[1])
fast_rows = np.concatenate([E.egg_aligned(f, [x["egg"] for x in FE[g] if x["fly"] == f["path"]]) for g in (T4, T2, TB) for f in G[g]])
ax.plot(tt, fast_rows.mean(0), label=f"fast eggs ({len(fast_rows)})", color="m"); ax.set_xlabel("s from egg"); ax.set_ylabel("mm/s"); ax.legend(fontsize=6)
plt.show()
print("Range of plotted mean speeds (mm/s):", round(float(fast_rows.mean(0).min()), 3), "to", round(float(fast_rows.mean(0).max()), 3))
allp = np.concatenate([pz[g][np.isfinite(pz[g])] for g in (T4, T2, TB)])
print(f"Pause times are multiples of {E.DT} s (one frame) in {np.mean(np.isclose(allp / E.DT, np.round(allp / E.DT))):.0%} of {len(allp)} eggs; "
      f"medians differ by {[round((np.median(pf_[g]) - np.median(pz[g][np.isfinite(pz[g])])) / E.DT) for g in (T4, T2, TB)]} frames (reading a), "
      "so the 10% bound holds only when the medians are equal")
crit3f = all(t3f[t3f.reading == "(a) own chamber"].p >= 0.05) and all(t3f[t3f.reading == "(a) own chamber"]["diff %"].abs() < 10)
crit3f_b = all(t3f[t3f.reading == "(b) pooled fast"].p >= 0.05) and all(t3f[t3f.reading == "(b) pooled fast"]["diff %"].abs() < 10)
print("Criterion (no chamber differs at p < 0.05 and medians within 10%): reading (a)", crit3f, "; reading (b)", crit3f_b)

**Result: not reproduced.** Under reading (a), fast eggs of the 4-mm chamber have a median pause of
0.5 s against 1.0 s for all its eggs (p = 0.0014); in the 2-mm and bounce chambers the medians are
equal (1.0 s; p = 0.12 and 0.33). Under reading (b), the 102 pooled fast eggs (median 0.5 s) differ
from the eggs of every chamber (p = 0.0002 to 0.0045). Every pause time is a whole number of 0.5-s
frames and the medians are 0.5 to 1 s, so the 10% bound on the median holds only when the medians
are equal; the 4-mm difference is one frame. The claim fails on the rank sum test as well, which
does not depend on that bound. The egg-aligned averages hold 979, 918 and 670 eggs (legend 979, 918,
675).

## fig4a-shorter-middle-transits

> Because flies follow the walls, transits through middle islands are shorter (median ~4.5 s) than
> through edge islands (median ~13 s).

In [ ]:
C3 = ["0-500-200-CS", "500-0-500-CS", "500-0-200-CS"]
TR = pd.DataFrame([dict(zip(["island", "dur", "frm", "to", "suc"], t), chamber=g) for g in C3 for f in G[g] for t in E.transits(f)])
TR["middle"] = TR.island == 1
TR["through"] = TR.frm != TR.to
rows = []
for reading, sub in (("(a) every visit", TR), ("(b) middle through, edges return", TR[(TR.middle & TR.through) | (~TR.middle)])):
    rows.append({"reading": reading, "chambers": "(i) pooled", "median middle (s)": sub[sub.middle].dur.median(),
                 "median edge (s)": sub[~sub.middle].dur.median(), "n middle": int(sub.middle.sum()), "n edge": int((~sub.middle).sum())})
    for g in C3:
        s = sub[sub.chamber == g]
        rows.append({"reading": reading, "chambers": g, "median middle (s)": s[s.middle].dur.median(),
                     "median edge (s)": s[~s.middle].dur.median(), "n middle": int(s.middle.sum()), "n edge": int((~s.middle).sum())})
t4a = pd.DataFrame(rows)
print(t4a.to_string(index=False))
print("\nTransit counts by sucrose, reading (b):")
sub = TR[(TR.middle & TR.through) | (~TR.middle)]
print(sub.groupby(["chamber", "suc"]).size().to_string())
print("Paper: 0/500/200 2637, 5694, 2903 (0, 500, 200); 500/0/500 1625, 1508 (0, 500); 500/0/200 1741, 3354, 1810 (500, 0, 200)")
print("Range of transit durations (s):", TR.dur.min(), "to", TR.dur.max())
print("Medians in frames (0.5 s each), pooled, reading (a): middle", TR[TR.middle].dur.median() / E.DT, "edge", TR[~TR.middle].dur.median() / E.DT,
      "; the tolerance on 4.5 s (4.05 to 4.95 s) holds 9 frames only")
ok4a = {(r["reading"], r["chambers"]): bool(abs(r["median middle (s)"] - 4.5) <= 0.45 and abs(r["median edge (s)"] - 13) <= 1.3) for _, r in t4a.iterrows()}
print("Both medians within tolerance:", ok4a)

**Result: partially reproduced.** Pooled over the three chambers, transits through the middle
island have a median of 4.0 s and through the edge islands 13.0 s, under either reading of a
transit. The edge median is within tolerance of 13 s; the middle median, 8 frames, is one frame
(0.5 s) below the tolerance of 4.05 to 4.95 s around 4.5 s, which holds only the value of 9 frames.
Per chamber, both medians are within tolerance in 500/0/500 (4.5 and 12.5 s) and 500/0/200 (4.5 and
13.5 s) but not in 0/500/200 (4.0 and 13.0 s), so reading (ii) holds in two chambers of three. With
middle transits counted only when they pass through (reading b), the 0/500/200 chamber has 2654,
5716 and 2993 transits on 0, 500 and 200 mM, against the legend's 2637, 5694 and 2903; the 0 mM
count is within tolerance.

## fig4b-faster-middle-preferred

> When the preferred 0 mM substrate is in the middle island, flies lay eggs more quickly after
> entering it than in three-island chambers with the preferred substrate at an edge.

The rate on 0 mM is taken by time since leaving the other substrate. In 500/0/200 the 0 mM island
has two neighbours, so the main reading takes the time since leaving either (the time since
entering 0 mM), and the curves since leaving 500 mM and since leaving 200 mM are shown as variants.

In [ ]:
EDGE3 = ["0-200-200-CS", "0-500-200-CS", "0-500-500-CS"]
MID = ["500-0-500-CS", "500-0-200-CS"]


def eval_4b(mode="working", show=False):
    rows = []
    for i, bn in ((1, "(a) 5-10"), (0, "(b) 0-5"), (2, "(c) 10-15")):
        for reading in ("visit", "search"):
            key = "trials_visit" if reading == "visit" else "trials_search"
            for m in MID:
                rm = R_entry(m, 0, mode)
                comps = {"(i) pooled edge chambers": R_entry(tuple(EDGE3), 0, mode)} | {"(ii) " + g: R_entry(g, 0, mode) for g in EDGE3}
                for cname, rc in comps.items():
                    rows.append({"bin": bn, "trials": reading, "middle chamber": m, "against": cname,
                                 "rate middle": round(rm["rate"][i], 3), "rate against": round(rc["rate"][i], 3),
                                 "p": E.ranksum01(rm[key][i], rc[key][i])})
    t = pd.DataFrame(rows)
    t["higher, p < 0.001"] = (t["rate middle"] > t["rate against"]) & (t.p < 0.001)
    verdict = {}
    for (bn, reading), s in t.groupby(["bin", "trials"]):
        pooled_ok = s[s.against.isin(["(i) pooled edge chambers", "(ii) 0-500-500-CS"])].groupby("middle chamber")["higher, p < 0.001"].all()
        each_ok = s[s.against.str.startswith("(ii)")].groupby("middle chamber")["higher, p < 0.001"].all()
        verdict[(bn, reading, "(i)")] = bool(pooled_ok.all()); verdict[(bn, reading, "(ii)")] = bool(each_ok.all())
    return t, verdict


t4b, v4b = eval_4b()
print(t4b.to_string(index=False))
print("\nReadings under which both middle chambers are higher at p < 0.001 against the edge chambers and against 0/500/500:")
for k, v in v4b.items():
    print("  ", k, v)
print("\nVariants by time since leaving a named substrate (5-10 s bin, visit trials):")
for m, prev in (("500-0-500-CS", 500), ("500-0-200-CS", 500), ("500-0-200-CS", 200)):
    rm = R(m, 0, prev)
    for g, pv in (("0-200-200-CS", 200), ("0-500-200-CS", 500), ("0-500-500-CS", 500)):
        rc = R(g, 0, pv)
        print(f"   {m} 0 since {prev}: {rm['rate'][1]:.3f} against {g} 0 since {pv}: {rc['rate'][1]:.3f}, p = {E.ranksum01(rm['trials_visit'][1], rc['trials_visit'][1]):.3g}")
fig, ax = plt.subplots(figsize=(6, 3))
plot_curves(ax, {m: R_entry(m) for m in MID} | {"edge chambers pooled": R_entry(tuple(EDGE3))}, "0 mM, time since entering"); plt.show()

**Result: reproduced.** Under reading (a), the 5 to 10 s bin, the rate on 0 mM after entering it is
1.816 eggs/min in 500/0/500 and 1.443 in 500/0/200, against 0.446 in the pooled edge chambers and
0.379 in 0/500/500, every comparison at p < 1e-07, with visit or search trials and with the edge
chambers pooled or taken one by one. Reading (c), 10 to 15 s, also holds on visit trials; reading
(b), 0 to 5 s, does not (no comparison significant). The curves since leaving a named neighbour
give the same result in the 5 to 10 s bin.

## fig4d-anomalous-eggs-alternative

> With 0 mM in the middle (500/0/200), flies also have a higher egg-laying rate than in other
> three-choice chambers on the closest alternative, 200 mM, from 0 to 90 s after leaving 0 mM.

In [ ]:
for reading in ("visit", "search"):
    a = E.pooled_trials(R("500-0-200-CS", 200, 0), 0, 90, reading); b = E.pooled_trials(R("0-500-200-CS", 200, 0), 0, 90, reading)
    print(f"{reading} trials: 500/0/200 {a[0]:.3f} eggs/min ({a[2]} eggs, {len(a[1])} trials); 0/500/200 {b[0]:.3f} ({b[2]} eggs, {len(b[1])} trials); p = {E.ranksum01(a[1], b[1]):.3g}")
print(curve_table({"500/0/200": R("500-0-200-CS", 200, 0), "0/500/200": R("0-500-200-CS", 200, 0)}, [("500/0/200", "0/500/200")]).iloc[:10].to_string(index=False))

**Result: reproduced.** Over 0 to 90 s after leaving 0 mM, the rate on 200 mM is 0.407 eggs/min in
500/0/200 (288 eggs) against 0.142 in 0/500/200 (117 eggs), p = 4.0e-29 on visit trials and
5.0e-21 on search trials.

## fig4e-inter-egg-not-longer

> The interval between eggs is not increased when the preferred substrate is in the middle island
> (medians 5.1, 4.1 and 4.4 min over 2262, 922 and 1901 intervals), so flies are not abandoning
> searches there.

In [ ]:
IEI = {g: np.concatenate([np.diff(f["eggs"]) * E.DT / 60 for f in G[g] if len(f["eggs"]) > 1]) for g in C3}
t4e = pd.DataFrame([{"chamber": g, "intervals": len(IEI[g]), "median (min)": round(float(np.median(IEI[g])), 2),
                     "eggs": sum(len(f["eggs"]) for f in G[g]), "flies with >= 2 eggs": sum(len(f["eggs"]) > 1 for f in G[g])} for g in C3])
print(t4e.to_string(index=False))
print("Paper, in the order of Fig. 4A: medians 5.1, 4.1, 4.4 min; 2262, 922, 1901 intervals; Fig. 4F eggs 2312, 943, 1946")
for m in MID:
    print(f"{m} against 0/500/200: rank sum p = {E.ranksum(IEI[m], IEI['0-500-200-CS']):.3g}")
print("Per fly median interval (min), median over flies:", {g: round(float(np.median([np.median(np.diff(f['eggs'])) * E.DT / 60 for f in G[g] if len(f['eggs']) > 1])), 2) for g in C3})
exp4e = [(5.1, 0.51), (4.1, 0.41), (4.4, 0.44)]; expn = [(2262, 226.2), (922, 92.2), (1901, 190.1)]
inside = [abs(m - v) <= t for m, (v, t) in zip(t4e["median (min)"], exp4e)] + [abs(n - v) <= t for n, (v, t) in zip(t4e["intervals"], expn)]
not_longer = all(np.median(IEI[m]) <= np.median(IEI["0-500-200-CS"]) for m in MID)
print("Criterion: six values within tolerance", inside, "; neither middle-preferred chamber longer:", not_longer, "->", "reproduced" if all(inside) and not_longer else "not reproduced")

**Result: reproduced.** In the order of Fig. 4A (0/500/200, 500/0/500, 500/0/200), the files give
2260, 922 and 1901 intervals between eggs with medians of 5.09, 4.38 and 4.12 min; all six values
are within tolerance of 2262, 922, 1901 and 5.1, 4.1, 4.4 min. The egg counts, 2311, 943 and 1946,
match Fig. 4F's 2312, 943 and 1946 in the same order. Neither middle-preferred chamber has a longer
median interval than 0/500/200; both are shorter (p = 0.008 and 2.1e-08).

## fig4f-motor-program-not-accelerated

> With the preferred substrate in the middle, the egg deposition motor program, proxied by the dip
> in locomotor speed, is not clearly accelerated.

In [ ]:
PZ = {g: np.concatenate([f["pause"] for f in G[g]]) for g in G}
ref = PZ["0-500-200-CS"]
for g in C3:
    x = PZ[g][np.isfinite(PZ[g])]
    print(f"{g}: pause median {np.median(x):.2f} s, mean {x.mean():.2f}, n {len(x)}; egg-aligned eggs (+-{E.ALIGN_S} s) {aligned_count([g])}"
          + ("" if g == "0-500-200-CS" else f"; against 0/500/200 p = {E.ranksum(x, ref):.3g}, median change {100 * (np.median(x) - np.median(ref)) / np.median(ref):.0f}%"))
print("Pause time distribution (s), 0/500/200:", np.percentile(ref[np.isfinite(ref)], [10, 25, 50, 75, 90]).tolist())
ok4f = {}
for g in MID:
    x = PZ[g][np.isfinite(PZ[g])]; r_ = ref[np.isfinite(ref)]
    shorter_sig = np.median(x) < np.median(r_) and E.ranksum(x, r_) < 0.05
    ok4f[g] = (not shorter_sig) or (np.median(r_) - np.median(x)) / np.median(r_) < 0.10
print("Criterion (not shorter at p < 0.05, or shorter by less than 10%):", ok4f, "->", "reproduced" if all(ok4f.values()) else "not reproduced")

**Result: reproduced.** The median pause is 1.0 s in all three chambers; neither middle-preferred
chamber differs from 0/500/200 (p = 0.56 and 0.20). The egg-aligned average of 0/500/200 holds
2310 eggs (legend 2312).

## fig5ab-strains-release-faster

> The DL and TUT wild-type strains release inhibition of egg laying on 200 mM faster than CS in
> 0/200 mM chambers; TUT shows no detectable effect of substrate history after about 20 s on a new
> substrate, while the CS curves do not merge within the plotted interval.

In [ ]:
def merges(groups, mode="working"):
    rows = []
    for g in groups:
        r0, r2 = R(g, 0, 200, mode=mode), R(g, 200, 0, mode=mode)
        rows.append({"group": g, **{f"{rd}, first >= {fl}": merge(r0, r2, rd, fl) for rd in ("visit", "search") for fl in (5, 10)}})
    return pd.DataFrame(rows).set_index("group")


def eval_5ab(mode="working", reading="visit", first_left=5):
    m = {g: merge(R(g, 0, 200, mode=mode), R(g, 200, 0, mode=mode), reading, first_left) for g in ["0-200-TUT", "0-200-DL", "0-200-CS"]}
    tut, dl, cs = m["0-200-TUT"], m["0-200-DL"], m["0-200-CS"]
    ok = (tut is not None and 15 <= tut <= 25) and (dl is not None and (cs is None or dl < cs)) and cs is None
    return ok, (tut, dl, cs)


m5 = merges(["0-200-TUT", "0-200-DL", "0-200-CS", "0-200-suzukii"])
print("Merge points (left edge of the first bin with p > 0.05; None: no merge within 7200 s):")
print(m5.to_string())
for reading in ("visit", "search"):
    for fl in (5, 10):
        print(f"{reading} trials, first bin >= {fl} s: criterion holds, (TUT, DL, CS) =", eval_5ab(reading=reading, first_left=fl))
for g in ["0-200-TUT", "0-200-DL", "0-200-CS"]:
    print(g, "p per bin:", np.round(E.bin_pvals(R(g, 0, 200), R(g, 200, 0)), 3).tolist())
fig, axs = plt.subplots(1, 3, figsize=(13, 3))
for ax, g in zip(axs, ["0-200-CS", "0-200-DL", "0-200-TUT"]):
    plot_curves(ax, {"0 since 200": R(g, 0, 200), "200 since 0": R(g, 200, 0)}, g)
plt.tight_layout(); plt.show()
print("Adopted reading (assumption): visit trials, since the Methods treat 'each substrate transition within a search' as independent "
      "(Calculation of egg-laying rate); with it and the 5-10 s bin first the criterion holds:", eval_5ab()[0])

**Result: reproduced.** With visit trials, TUT merges at 20 s and DL at 90 s, while CS does not merge
within 7200 s (every bin from 5 s on is at p < 0.05), whichever first bin is taken. With search
trials, reading (b) of M5, TUT merges at 90 s, outside the tolerance, and the claim would not be
reproduced. The outcome rests on the visit reading, adopted as an assumption: the Methods say the
P values "assume that each individual search and each substrate transition within a search are
independent" (Calculation of egg-laying rate), which makes a transition, a visit, the trial.

## fig5c-suzukii-no-preference

> D. suzukii shows no measurable egg-laying preference for lower sucrose; its rates on 0 and 200 mM
> are similar in a two-choice chamber.

In [ ]:
def eval_5c(mode="working", reading="visit"):
    r0, r2 = R("0-200-suzukii", 0, 200, mode=mode), R("0-200-suzukii", 200, 0, mode=mode)
    p = E.bin_pvals(r0, r2, reading)
    sig = [BINS[i] for i in range(len(BINS)) if EDGES[i] >= 5 and p[i] < 0.05 and r0["rate"][i] > r2["rate"][i]]
    return len(sig) == 0, sig


r0s, r2s = R("0-200-suzukii", 0, 200), R("0-200-suzukii", 200, 0)
t5c = curve_table({"0 since 200": r0s, "200 since 0": r2s}, [("0 since 200", "200 since 0")])
t5c["difference (0 - 200)"] = np.round(r0s["rate"] - r2s["rate"], 3)
t5c["p search"] = np.round(E.bin_pvals(r0s, r2s, "search"), 4)
print(t5c.to_string(index=False))
print("Bins past 5 s with 0 mM significantly higher (visit trials):", eval_5c()[1], "; (search trials):", eval_5c(reading="search")[1])
past5 = [i for i in range(len(BINS)) if EDGES[i] >= 5]
for reading in ("visit", "search"):
    p_ = E.bin_pvals(r0s, r2s, reading)[past5]
    bh = stats.false_discovery_control(np.nan_to_num(p_, nan=1.0), method="bh")
    m_ = len(p_); order = np.argsort(p_); holm_adj = np.empty(m_)
    holm_adj[order] = np.minimum(1, np.maximum.accumulate((m_ - np.arange(m_)) * np.nan_to_num(p_[order], nan=1.0)))
    print(f"Variant, {reading} trials, {m_} bins past 5 s: smallest p {np.nanmin(p_):.3g}; Holm-adjusted {holm_adj.min():.3g}; "
          f"Benjamini-Hochberg {bh.min():.3g}; bins significant after Holm: {int((holm_adj < 0.05).sum())} (not the claims file's criterion)")

**Result: not reproduced.** The claim is one of absence, judged on the bin test at p < 0.05. Of the
13 bins from 5 s on, one, 300 to 600 s, has a higher rate on 0 mM than on 200 mM at p = 0.029 (13 of
17 visits against 8 of 20; 0.94 against 0.55 eggs/min), under both trial readings. In the other 12
bins no difference reaches p < 0.05, and the differences change sign from bin to bin (from -0.47 to
+0.39 eggs/min); in the bins up to 300 s, where most eggs are, the 0 mM rate is not significantly
higher in any. The criterion counts any such bin, without a correction for testing 13 bins; with a
Holm correction the smallest adjusted p is 0.38 and no bin is significant, which is what the paper's
statement would lead one to expect. That variant is not the claims file's criterion, and the
question of whether the criterion should correct for the number of bins is put to a reader.

## fig5de-suzukii-more-on-200

> D. suzukii lays slightly more eggs on 200 mM than on 0 mM, consistent with spending more time on
> high sucrose.

In [ ]:
def egg_frac(groups, value=0):
    groups = [groups] if isinstance(groups, str) else groups
    k = sum(int((f["sucrose"][f["eggs"]] == value).sum()) for g in groups for f in G[g])
    n = sum(len(f["eggs"]) for g in groups for f in G[g])
    return E.frac_ci(k, n), k, n


def time_frac(g, value=0, search_only=False):
    out = []
    for f in G[g]:
        m = E.in_search_mask(f) if search_only else np.ones(f["n"], bool)
        out.append(np.mean(f["sucrose"][m] == value) if m.any() else np.nan)
    return np.array(out)


(fr, lo, hi), k, n = egg_frac("0-200-suzukii")
tf = time_frac("0-200-suzukii")
print(f"D. suzukii: {k} of {n} eggs on 0 mM, fraction {fr:.3f} (95% CI {lo:.3f} to {hi:.3f}); flies {len(G['0-200-suzukii'])}")
print(f"Per-fly fraction of time on 0 mM: median {np.median(tf):.3f}, range {tf.min():.3f} to {tf.max():.3f}; flies below 0.5: {(tf < 0.5).sum()} of {len(tf)}")
pfe = [np.mean(f["sucrose"][f["eggs"]] == 0) for f in G["0-200-suzukii"] if len(f["eggs"])]
print(f"Per-fly fraction of eggs on 0 mM (flies with eggs, {len(pfe)}): median {np.median(pfe):.3f}")

**Result: reproduced.** D. suzukii lay 85 of 220 eggs on 0 mM (0.386, 95% CI 0.322 to 0.454), and
the median fly spends 0.197 of its time on 0 mM (25 of 29 flies below 0.5).

## fig5f-egg-numbers-differ

> Different strains and species lay different numbers of eggs.

In [ ]:
S5 = {"D. suzukii": "0-200-suzukii", "TUT": "0-200-TUT", "DL": "0-200-DL", "CS": "0-200-CS"}
EPF = {k: np.array([len(f["eggs"]) for f in G[g]]) for k, g in S5.items()}
print({k: f"mean {v.mean():.1f}, median {np.median(v):.0f}, n {len(v)} flies" for k, v in EPF.items()})
for a, b in combinations(S5, 2):
    print(f"{a} against {b}: rank sum p = {E.ranksum(EPF[a], EPF[b]):.3g}")

**Result: reproduced.** Mean eggs per fly are 7.6 (D. suzukii), 21.9 (TUT), 32.0 (DL) and 39.6 (CS),
and all six pairs differ at p < 0.05 (largest p 0.011, DL against CS).

## fig5gh-search-and-pause

> All strains and species raise their locomotor speed before pausing to lay an egg, and they differ
> from one another in the delay between the speed dip and egg deposition.

In [ ]:
def pause_pairs(named):
    rows = []
    for a, b in combinations(named, 2):
        x, y = PZ[named[a]], PZ[named[b]]
        rows.append({"pair": f"{a} / {b}", "median a": np.nanmedian(x), "median b": np.nanmedian(y), "mean a": round(np.nanmean(x), 2),
                     "mean b": round(np.nanmean(y), 2), "p": E.ranksum(x, y)})
    return pd.DataFrame(rows)


def rise(named):
    rows = []
    for k, g in named.items():
        sr = np.concatenate([E.speed_rise(f) for f in G[g]])
        rows.append({"group": k, "eggs": len(sr), "speed 60 s before pause": round(sr[:, 0].mean(), 3),
                     "speed 60 s before that": round(sr[:, 1].mean(), 3), "rises": bool(sr[:, 0].mean() > sr[:, 1].mean()),
                     "paired Wilcoxon p": stats.wilcoxon(sr[:, 0], sr[:, 1]).pvalue, "egg-aligned eggs": aligned_count([g])})
    return pd.DataFrame(rows)


t5h = pause_pairs(S5)
print(t5h.to_string(index=False))
print("Pairs at p < 0.001:", int((t5h.p < 0.001).sum()), "of 6")
print(rise(S5).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
for k, g in S5.items():
    ax.plot(np.arange(-E.ALIGN_S * 2, E.ALIGN_S * 2 + 1) * E.DT, np.concatenate([E.egg_aligned(f) for f in G[g]]).mean(0), label=k)
ax.legend(fontsize=6); ax.set_xlabel("s from egg"); ax.set_ylabel("mm/s"); plt.show()

**Result: reproduced.** All six pairs of strains and species differ in pause time at p < 0.001
(largest p 6.1e-04, TUT against CS; medians 4.5, 1.0, 2.5 and 1.0 s for D. suzukii, TUT, DL and CS).
In all four, mean speed over the 60 s before the pause is higher than over the 60 s before that
(0.63 against 0.45, 1.03 against 0.79, 0.84 against 0.76 and 0.81 against 0.62 mm/s). The
egg-aligned averages hold 216, 1184, 767 and 1863 eggs, within tolerance of 214, 1184, 766 and 1862.

## fig6cd-hardness-fast-merge

> Unlike sucrose, flies do not strongly repress egg laying for minutes on the harder option; the 0.8%
> and 1.4% rate curves merge within about 20 s, also with equal sucrose on both.

The rate on 0.8% since leaving 1.4% and on 1.4% since leaving 0.8% (M4), with the agarose of each
frame from the `Agarose_Percentage` series, in the 0.8/1.4% chamber and the one chamber with equal
sucrose on both sides that the dandiset holds (200 mM).

In [ ]:
AG = ["0p8--1p4--CS", "0p8-200-1p4-200-CS"]
rows = []
for g in AG:
    r8, r14 = R(g, 8, 14, key="agarose"), R(g, 14, 8, key="agarose")
    for reading in ("visit", "search"):
        for fl in (5, 10):
            rows.append({"chamber": g, "trials": reading, "first bin": f">= {fl} s", "merge (s)": merge(r8, r14, reading, fl)})
    print(curve_table({"0.8 since 1.4": r8, "1.4 since 0.8": r14}, [("0.8 since 1.4", "1.4 since 0.8")]).to_string(index=False))
    print(f"  {g}: bins past 5 s at p < 0.05 after the merge (visit trials):",
          [BINS[i] for i, p in enumerate(E.bin_pvals(r8, r14)) if EDGES[i] >= 5 and p < 0.05 and EDGES[i] > (merge(r8, r14) or 0)])
t6cd = pd.DataFrame(rows)
print(t6cd.to_string(index=False))
for (reading, fl), s in t6cd.groupby(["trials", "first bin"]):
    print(f"Criterion, {reading} trials, first bin {fl}: every chamber merges at or before 25 s:",
          bool(all(m is not None and m <= 25 for m in s["merge (s)"])))
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, g in zip(axs, AG):
    plot_curves(ax, {"0.8 since 1.4": R(g, 8, 14, key="agarose"), "1.4 since 0.8": R(g, 14, 8, key="agarose")}, g)
plt.tight_layout(); plt.show()
print("Plotted rates range:", round(float(min(np.nanmin(R(g, a, b, key="agarose")["rate"]) for g in AG for a, b in ((8, 14), (14, 8)))), 3), "to",
      round(float(max(np.nanmax(R(g, a, b, key="agarose")["rate"]) for g in AG for a, b in ((8, 14), (14, 8)))), 3), "eggs/min")

**Result: reproduced.** The 0.8% and 1.4% rate curves merge at 20 s in the 0.8/1.4% chamber, under
both trial readings and both first-bin readings, and at 20 s (visit trials) or 25 s (search trials)
in the chamber with 200 mM on both sides; every merge is at or before 25 s. Before the merge the
0.8% rate is higher (5 to 10 s: 0.725 against 0.071 eggs/min without sucrose, 0.439 against 0.193
with). Some later bins differ again at p < 0.05 (30 to 45, 150 to 300 and 300 to 600 s without
sucrose; 90 to 150 and 300 to 600 s with), which the merge-point definition does not count. The
dandiset holds one of the paper's equal-sucrose chambers, so "also with equal sucrose" is tested on
that one.

## fig6ef-pause-longer-hard

> Flies pause longer to lay eggs on harder substrates, and egg-laying rates are lower for about 15 s
> after crossing the boundary in a 1.4%/1.4% chamber than in a 0.8%/0.8% chamber.

With the same substrate on both sides, the substrate is the island; the rate is taken since
crossing from the other island, both directions pooled.

In [ ]:
def both_dirs(g):
    a, b = R(g, 0, 1, key="island"), R(g, 1, 0, key="island")
    out = {k: a[k] + b[k] for k in ("eggs", "frames", "visits")}
    out["rate"] = out["eggs"] / out["frames"] * 120
    out["trials_visit"] = [np.concatenate([x, y]) for x, y in zip(a["trials_visit"], b["trials_visit"])]
    out["trials_search"] = [np.concatenate([x, y]) for x, y in zip(a["trials_search"], b["trials_search"])]
    out["edges"] = EDGES
    return out


p08, p14 = PZ["0p8--0p8--CS"], PZ["1p4--1p4--CS"]
print(f"Pause 0.8/0.8: median {np.nanmedian(p08):.2f} s, mean {np.nanmean(p08):.2f}, n {np.isfinite(p08).sum()}; "
      f"1.4/1.4: median {np.nanmedian(p14):.2f}, mean {np.nanmean(p14):.2f}, n {np.isfinite(p14).sum()}; p = {E.ranksum(p08, p14):.3g}")
print("Pause percentiles 10/25/50/75/90 (s): 0.8", np.nanpercentile(p08, [10, 25, 50, 75, 90]).tolist(), "; 1.4", np.nanpercentile(p14, [10, 25, 50, 75, 90]).tolist())
print("Egg-aligned eggs:", aligned_count(["0p8--0p8--CS"]), aligned_count(["1p4--1p4--CS"]), "(paper 831, 795)")
r08, r14 = both_dirs("0p8--0p8--CS"), both_dirs("1p4--1p4--CS")
t6 = curve_table({"0.8/0.8": r08, "1.4/1.4": r14}, [("1.4/1.4", "0.8/0.8")])
t6["p search"] = np.round(E.bin_pvals(r14, r08, "search"), 4)
print(t6.iloc[:8].to_string(index=False))
lower15 = all(r14["rate"][i] < r08["rate"][i] for i in (0, 1, 2))
sig15 = [BINS[i] for i in (0, 1, 2) if r14["rate"][i] < r08["rate"][i] and E.bin_pvals(r14, r08)[i] < 0.05]
print("1.4 lower in all bins up to 15 s:", lower15, "; significant bins:", sig15)
pv6 = E.bin_pvals(r14, r08)
lower_past5 = all(r14["rate"][i] < r08["rate"][i] for i in (1, 2))
print(f"Criterion: pause longer on 1.4% at p < 0.001: {bool(np.nanmedian(p14) > np.nanmedian(p08) and E.ranksum(p08, p14) < 0.001)}; "
      f"1.4% rate lower in every bin up to 15 s: {lower15} (the 0-5 s bin is 0 in both, a tie); lower in the 5-10 and 10-15 s bins: {lower_past5}; "
      f"at least one significant: {len(sig15) > 0}")

**Result: reproduced.** Pause times are longer on 1.4% than on 0.8% agarose (median 1.5 against 1.0
s, p = 4.1e-21; the mean is lower on 1.4%, 1.99 against 4.94 s, because of a few long pauses on 0.8%).
After crossing the barrier, the 1.4% rate is lower than the 0.8% rate in the 5 to 10 s bin (0.144
against 0.542 eggs/min, p < 0.0001) and the 10 to 15 s bin (0.610 against 1.183, p = 0.0002), and
the two are close from 15 s on. In the 0 to 5 s bin neither chamber has an egg (0 of 973 and 0 of
1478 visits), so the rate there cannot be lower; this tie is broken as the paper's merge-point
definition does, by leaving out the bins before 5 s. Read literally, with that bin, the criterion
fails. The egg-aligned averages hold 831 and 795 eggs, as in the legend.

## fig6g-more-time-softer

> Flies spend more time on the softer option in 0.8% against 1.4% chambers.

Per-fly fraction of time on 0.8% agarose, from the `Agarose_Percentage` series, over all frames and over
search frames (M3, no 30-s minimum).

In [ ]:
for g in ["0p8--1p4--CS"]:
    tf_all = np.array([np.mean(f["agarose"] == 8) for f in G[g]])
    tf_s = np.array([np.mean(f["agarose"][E.in_search_mask(f)] == 8) for f in G[g] if len(f["eggs"])])
    for name, x in (("all frames", tf_all), ("search frames", tf_s)):
        print(f"{g}, {name}: median fraction of time on 0.8% {np.median(x):.3f} over {len(x)} flies (range {x.min():.3f} to {x.max():.3f}); "
              f"flies above 0.5: {(x > 0.5).sum()}; median above 0.5: {np.median(x) > 0.5}")
    orient = np.array([int(f["agarose"][f["island"] == 0][0]) for f in G[g]])
    for a in (8, 14):
        print(f"  flies with island 0 at {a / 10}%: median fraction of time on 0.8% {np.median(tf_all[orient == a]):.3f} ({(orient == a).sum()} flies)")
g2 = "0p8-200-1p4-200-CS"
x2 = np.array([np.mean(f["agarose"] == 8) for f in G[g2]])
print(f"Variant, {g2} (200 mM on both): median {np.median(x2):.3f} over {len(x2)} flies")

**Result: reproduced.** In the 0.8/1.4% chamber the median fraction of time on 0.8% agarose is 0.609
over all frames (24 flies, 18 above 0.5) and 0.672 over search frames (23 flies with eggs, 22 above
0.5). It is above 0.5 whichever way round the chamber was loaded (0.562 for the 18 flies with 0.8% on
island 0, 0.763 for the 6 with 1.4% there).

## fig7cf-kir-merge-earlier

> The more Kir2.1 is expressed in DDC-Gal4 neurons (pretreatment at 18, 27, 29/29.5 and 31 C), the
> earlier the 0 and 200 mM rate curves merge; at 31 C they are nearly superimposable throughout,
> while at the two intermediate temperatures rates stay properly biased just after a transition.

In [ ]:
DDC = {"18": "0-200-DDC-18C", "27": "0-200-DDC-27C", "29/29.5": "0-200-DDC-29CAND295C", "31": "0-200-DDC-31C"}


def eval_7cf(mode="working", reading="visit", first_left=5):
    m = {}
    for t, g in DDC.items():
        v = merge(R(g, 0, 200, mode=mode), R(g, 200, 0, mode=mode), reading, first_left)
        m[t] = INF if v is None else v
    mono = m["18"] >= m["27"] >= m["29/29.5"] >= m["31"] and m["31"] < m["18"]
    p31 = E.bin_pvals(R(DDC["31"], 0, 200, mode=mode), R(DDC["31"], 200, 0, mode=mode), reading)
    nsig31 = int(sum(p31[i] < 0.05 for i in range(len(BINS)) if EDGES[i] >= 5))
    ra = m["31"] == float(first_left)
    rb = nsig31 <= 1
    mid = {}
    for t in ("27", "29/29.5"):
        r0, r2 = R(DDC[t], 0, 200, mode=mode), R(DDC[t], 200, 0, mode=mode)
        p = E.bin_pvals(r0, r2, reading)
        mid[t] = any(p[i] < 0.05 and r0["rate"][i] > r2["rate"][i] for i in bins_in(5, 30))
    ok = mono and (ra or rb) and all(mid.values())
    return ok, {"merge": m, "monotone": mono, "31C merge first bin": ra, "31C bins past 5 s at p<0.05": nsig31, "biased 27, 29": mid}


for reading in ("visit", "search"):
    for fl in (5, 10):
        print(reading, f"first >= {fl} s:", eval_7cf(reading=reading, first_left=fl))
for t, g in DDC.items():
    print(t, "C p per bin:", np.round(E.bin_pvals(R(g, 0, 200), R(g, 200, 0)), 3).tolist())
fig, axs = plt.subplots(1, 4, figsize=(16, 3))
for ax, (t, g) in zip(axs, DDC.items()):
    plot_curves(ax, {"0 since 200": R(g, 0, 200), "200 since 0": R(g, 200, 0)}, f"DDC>Kir {t} C")
plt.tight_layout(); plt.show()
print("Adopted readings (assumptions): visit trials, and the 5-10 s bin as the first bin past 5 s. Outcome under them:",
      "reproduced" if eval_7cf()[0] else ("partially reproduced" if eval_7cf()[1]["monotone"] else "not reproduced"),
      "; with the 10-15 s bin first:", "reproduced" if eval_7cf(first_left=10)[0] else "not reproduced")

**Result: partially reproduced.** The merge points fall with pretreatment temperature: no merge
within 7200 s at 18 C, 600 s at 27 C, 25 s at 29/29.5 C and 10 s at 31 C (visit trials), and at 27
and 29/29.5 C the 0 mM rate is higher than the 200 mM rate at p < 0.05 in bins between 5 and 30 s.
At 31 C two bins past 5 s differ at p < 0.05 (5 to 10 s, p = 0.001, and 45 to 60 s, p = 0.007),
so reading (b) of "nearly superimposable" fails, and under the first-bin reading taken here (the
5 to 10 s bin is the first bin past 5 s) the merge point, 10 s, is not the first bin, so reading
(a) fails too. If the first bin past 5 s is taken as 10 to 15 s, reading (a) holds and the claim is
reproduced, under visit and search trials alike. The paper's words do not settle which bin is
first; the 5 to 10 s bin is adopted as an assumption, as its left edge is 5 s, so that "past 5 s"
leaves out only the 0 to 5 s bin, where every rate is near 0. The outcome rests on that choice.

## fig7df-overall-rate-drops

> The overall egg-laying rate during search tends to fall as Kir2.1 expression increases, meaning
> flies search longer before laying an egg.

In [ ]:
rows = []
for t, g in DDC.items():
    r0, r2 = R(g, 0, 200), R(g, 200, 0)
    eggs = r0["eggs"].sum() + r2["eggs"].sum(); frames = r0["frames"].sum() + r2["frames"].sum()
    dur = np.concatenate([(f["eggs"] - f["starts"]) * E.DT for f in G[g]])
    rows.append({"temperature": t, "eggs": int(eggs), "search frames": int(frames), "overall rate (eggs/min)": round(eggs / frames * 120, 3),
                 "median search (s)": float(np.median(dur)), "flies": len(G[g])})
t7d = pd.DataFrame(rows)
print(t7d.to_string(index=False))
rt = t7d.set_index("temperature")["overall rate (eggs/min)"]
print("Decreasing from 27 to 29/29.5 to 31 C:", bool(rt["27"] > rt["29/29.5"] > rt["31"]))
# Variant: every egg over every frame of the working search windows, including frames before a fly's first
# crossing (which no rate function bin holds).
rows = []
for t, g in DDC.items():
    n_e = sum(len(f["eggs"]) for f in G[g])
    n_f = sum(int(sum(e - s + 1 for s, e in zip(*E.search_windows(f["starts"], f["eggs"], f["n"], "working")))) for f in G[g] if len(f["eggs"]))
    rows.append({"temperature": t, "eggs": n_e, "search frames": n_f, "overall rate (eggs/min)": round(n_e / n_f * 120, 3),
                 "eggs in the rate-function bins": int(t7d.set_index("temperature").loc[t, "eggs"])})
t7d2 = pd.DataFrame(rows)
print(t7d2.to_string(index=False))
rt2 = t7d2.set_index("temperature")["overall rate (eggs/min)"]
print("Variant decreasing from 27 to 29/29.5 to 31 C:", bool(rt2["27"] > rt2["29/29.5"] > rt2["31"]))

**Result: partially reproduced.** The overall rate during search is 0.557, 0.465, 0.208 and 0.219
eggs/min at 18, 27, 29/29.5 and 31 C, and the median search lasts 27.5, 49.5, 125.8 and 114.0 s.
The rate falls from 27 to 29/29.5 C but not from 29/29.5 to 31 C, so the decrease the criterion
asks for holds only in part. Counting every egg over every frame of the search windows, including
frames before a fly's first crossing that no rate-function bin holds (31 of 1031 eggs at 18 C), gives
0.567, 0.460, 0.208 and 0.220 eggs/min, the same pattern.

## fig7gh-choice-falls

> Increasing Kir2.1 induction progressively lowers the fraction of eggs on 0 mM, and DDC>Kir (31 C
> pre) flies lay more eggs on 200 than on 0 mM, consistent with spending more time on the 200 mM side.

In [ ]:
rows = []
for t, g in DDC.items():
    (fr, lo, hi), k, n = egg_frac(g)
    tf = time_frac(g)
    pfe = [np.mean(f["sucrose"][f["eggs"]] == 0) for f in G[g] if len(f["eggs"]) >= 5]
    rows.append({"temperature": t, "eggs on 0": k, "eggs": n, "fraction on 0": round(fr, 3), "95% CI": f"{lo:.3f} to {hi:.3f}",
                 "median time fraction on 0": round(float(np.median(tf)), 3), "flies": len(tf),
                 "flies >= 5 eggs": len(pfe), "median per-fly egg fraction": round(float(np.median(pfe)), 3)})
t7g = pd.DataFrame(rows)
print(t7g.to_string(index=False))
f7 = t7g["fraction on 0"].values
print("Pooled fraction decreasing 18 > 27 > 29/29.5 > 31:", bool(all(np.diff(f7) < 0)), "; 31 C below 0.5:", f7[-1] < 0.5,
      "; 31 C median time fraction below 0.5:", t7g["median time fraction on 0"].values[-1] < 0.5)
pf18 = [np.mean(f["sucrose"][f["eggs"]] == 0) for f in G[DDC["18"]] if len(f["eggs"]) >= 5]
pf31 = [np.mean(f["sucrose"][f["eggs"]] == 0) for f in G[DDC["31"]] if len(f["eggs"]) >= 5]
print(f"Per-fly fractions, flies with >= 5 eggs, 31 against 18 C: rank sum p = {E.ranksum(pf31, pf18):.3g}")

**Result: reproduced.** The fraction of eggs on 0 mM falls from 0.938 at 18 C to 0.919, 0.761 and
0.296 at 27, 29/29.5 and 31 C, below 0.5 at 31 C, and the median 31 C fly spends 0.266 of its time
on 0 mM.

## fig7ik-not-egg-number-or-motor

> The loss of preference in DDC>Kir flies is unrelated to the number of eggs laid or to changes in
> the egg deposition motor program.

In [ ]:
for t, g in DDC.items():
    e = np.array([len(f["eggs"]) for f in G[g]]); x = PZ[g]
    print(f"{t} C: eggs per fly mean {e.mean():.1f}, median {np.median(e):.0f} ({len(e)} flies); pause median {np.nanmedian(x):.2f} s, mean {np.nanmean(x):.2f}; egg-aligned eggs {aligned_count([g])}")
e18 = np.array([len(f["eggs"]) for f in G[DDC["18"]]]); e31 = np.array([len(f["eggs"]) for f in G[DDC["31"]]])
p_e = E.ranksum(e31, e18)
pm18, pm31 = np.nanmedian(PZ[DDC["18"]]), np.nanmedian(PZ[DDC["31"]])
print(f"Eggs per fly, 31 against 18 C: p = {p_e:.3g}; 31 C lower: {np.median(e31) < np.median(e18)}")
print(f"Median pause 31 C {pm31:.2f} s against 18 C {pm18:.2f} s: change {100 * (pm31 - pm18) / pm18:.0f}%; rank sum p = {E.ranksum(PZ[DDC['31']], PZ[DDC['18']]):.3g}")
s31 = [(len(f["eggs"]), np.mean(f["sucrose"][f["eggs"]] == 0)) for f in G[DDC["31"]] if len(f["eggs"]) >= 5]
rho = stats.spearmanr([a for a, _ in s31], [b for _, b in s31])
print(f"Variant: 31 C flies with >= 5 eggs ({len(s31)}), Spearman rho between eggs laid and fraction on 0 mM = {rho.statistic:.3f}, p = {rho.pvalue:.3g}")
print(f"The medians differ by {round((pm18 - pm31) / E.DT)} frames of 0.5 s; the 20% bound on a 2.5-s median is half a frame either side")

**Result: not reproduced.** DDC>Kir (31 C pre) flies lay fewer eggs per fly than 18 C controls
(mean 13.0 against 21.5, p = 2.4e-04), and their median pause is 1.5 s against 2.5 s, 40% shorter
(p = 9.2e-10), so neither part of the criterion holds. The pause medians differ by 2 frames of 0.5
s, where the 20% bound allows half a frame, but the rank sum test also finds them different. Within
31 C flies with at least 5 eggs, eggs laid and the fraction on 0 mM are not correlated (Spearman rho
= -0.07, p = 0.72), the variant; on that reading of "unrelated" the egg-number part would hold, and
whether the claims file's reading is the paper's is put to a reader. The pause part fails either
way. The egg-aligned averages hold 1030 and 533 eggs, within tolerance of 1029 and 533.

## fig8ce-mutants-flat-rates

> Dop1R1 and Dop1R2 mutants show similar egg-laying rate curves on 0 and 200 mM; the Dop1R1 curves
> resemble the CS 0 mM curve and the Dop1R2 curves the CS 200 mM curve.

In [ ]:
MUT = {"Dop1R1-": "0-200-Dop1R1-mutant", "Dop1R2 attP": "0-200-Dop1R2-mutant-1", "Dop1R2-": "0-200-Dop1R2-mutant-2"}
b300 = bins_in(5, 300)
cs0, cs2 = R("0-200-CS", 0, 200)["rate"][b300].mean(), R("0-200-CS", 200, 0)["rate"][b300].mean()
cs0p, cs2p = E.pooled_trials(R("0-200-CS", 0, 200), 5, 300)[0], E.pooled_trials(R("0-200-CS", 200, 0), 5, 300)[0]
print(f"CS mean rate over bins 5-300 s: 0 mM {cs0:.3f}, 200 mM {cs2:.3f} eggs/min (pooled eggs/frames: {cs0p:.3f}, {cs2p:.3f})")
rows = []
for k, g in MUT.items():
    r0, r2 = R(g, 0, 200), R(g, 200, 0)
    for reading in ("visit", "search"):
        p = E.bin_pvals(r0, r2, reading)
        sig = [BINS[i] for i in range(len(BINS)) if EDGES[i] >= 5 and p[i] < 0.05 and r0["rate"][i] > r2["rate"][i]]
        m0, m2 = r0["rate"][b300].mean(), r2["rate"][b300].mean()
        rows.append({"mutant": k, "trials": reading, "bins 0 > 200 at p<0.05 (past 5 s)": ", ".join(sig) or "none", "n": len(sig),
                     "mean 0 mM": round(m0, 3), "mean 200 mM": round(m2, 3),
                     "0 nearer CS 0": abs(m0 - cs0) < abs(m0 - cs2), "200 nearer CS 0": abs(m2 - cs0) < abs(m2 - cs2),
                     "pooled 0": round(E.pooled_trials(r0, 5, 300)[0], 3), "pooled 200": round(E.pooled_trials(r2, 5, 300)[0], 3)})
t8 = pd.DataFrame(rows)
print(t8.to_string(index=False))
fig, axs = plt.subplots(1, 3, figsize=(13, 3))
for ax, (k, g) in zip(axs, MUT.items()):
    plot_curves(ax, {"0 since 200": R(g, 0, 200), "200 since 0": R(g, 200, 0)}, k)
plt.tight_layout(); plt.show()

**Result: reproduced.** No mutant has more than one bin past 5 s where the 0 mM rate exceeds the 200
mM rate at p < 0.05 (Dop1R1- none, Dop1R2 attP one, Dop1R2- none). Over bins 5 to 300 s the mean
rates are 0.707 and 0.905 eggs/min for Dop1R1- (nearer the CS 0 mM curve, 1.199, than the CS 200
mM curve, 0.129), and 0.213 and 0.315 for Dop1R2 attP and 0.234 and 0.200 for Dop1R2- (nearer the
CS 200 mM curve).

## fig8fg-time-and-eggs

> Dop1R1- and Dop1R2 attP flies spend more time on 200 mM, and the three mutants lay a similar number
> of eggs and raise their speed before pausing to lay.

In [ ]:
for k, g in MUT.items():
    tf = time_frac(g); e = np.array([len(f["eggs"]) for f in G[g]])
    print(f"{k}: median time fraction on 0 mM {np.median(tf):.3f} ({len(tf)} flies, {np.sum(tf < 0.5)} below 0.5); eggs per fly mean {e.mean():.1f}, median {np.median(e):.0f}; egg-aligned eggs {aligned_count([g])}")
EM = {k: np.array([len(f["eggs"]) for f in G[g]]) for k, g in MUT.items()}
for a, b in combinations(MUT, 2):
    print(f"eggs per fly, {a} against {b}: p = {E.ranksum(EM[a], EM[b]):.3g}")
print(rise(MUT).to_string(index=False))
print("Speed rise with other windows (mean speed over the w s before the pause against the w s before that):")
for w in (30, 60, 120):
    for k, g in MUT.items():
        sr = np.concatenate([E.speed_rise(f, w) for f in G[g]])
        print(f"  w = {w} s, {k}: {sr[:, 0].mean():.3f} against {sr[:, 1].mean():.3f} mm/s ({len(sr)} eggs), rises: {bool(sr[:, 0].mean() > sr[:, 1].mean())}, "
              f"paired Wilcoxon p = {stats.wilcoxon(sr[:, 0], sr[:, 1]).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(6, 3))
tt = np.arange(-E.ALIGN_S * 2, E.ALIGN_S * 2 + 1) * E.DT
rng8 = []
for k, g in (MUT | {"CS": "0-200-CS"}).items():
    m = np.concatenate([E.egg_aligned(f) for f in G[g]]).mean(0); rng8 += [m.min(), m.max()]
    ax.plot(tt, m, label=k)
ax.legend(fontsize=6); ax.set_xlabel("s from egg"); ax.set_ylabel("mm/s (5-s boxcar)"); plt.show()
print(f"Range of the plotted mean speeds: {min(rng8):.3f} to {max(rng8):.3f} mm/s")
tf1, tf2 = np.median(time_frac(MUT["Dop1R1-"])), np.median(time_frac(MUT["Dop1R2 attP"]))
eggs_sim = all(E.ranksum(EM[a], EM[b]) >= 0.05 for a, b in combinations(MUT, 2))
rises = rise(MUT)["rises"].tolist()
print(f"Criterion: median time fractions below 0.5 ({tf1:.3f}, {tf2:.3f}): {tf1 < 0.5 and tf2 < 0.5}; no egg-count pair at p < 0.05: {eggs_sim}; "
      f"speed rises (60 s windows) in {sum(rises)} of 3")
EM_late = {k: np.array([int((f["eggs"] < f["n"] - LATE_S * E.FPS).sum()) for f in G[g]]) for k, g in MUT.items()}
print("Without eggs in the last 30 s of a recording (sanity check above), eggs per fly p:",
      {f"{a} / {b}": round(E.ranksum(EM_late[a], EM_late[b]), 3) for a, b in combinations(MUT, 2)})

**Result: partially reproduced.** Dop1R1- and Dop1R2 attP flies spend a median of 0.399 and 0.393 of
their time on 0 mM, and no pair of the three mutants differs in eggs per fly (p = 0.37 to 0.80; 0.34
to 0.69 without the eggs annotated in the last 30 s of a recording). Speed over the 60 s before the
pause is higher than over the 60 s before that in Dop1R1- (0.79 against 0.63 mm/s) but not in Dop1R2
attP (1.61 against 1.66) or Dop1R2- (1.94 against 2.06), so the speed rise holds in one mutant of
three. With 30-s windows it again rises only in Dop1R1-; with 120-s windows it rises in Dop1R1- and,
barely (2.00 against 1.97, p = 0.65), in Dop1R2-, not in Dop1R2 attP. The egg-aligned averages hold
498, 586 and 323 eggs, as in the legend.

## figs2-search-definition-robust

> The conclusions drawn from the rate functions hold under alternative definitions of the search
> period (no 30-s minimum; 90 s before each egg; 15 min before each egg; the last two shortened so as
> not to reach back past the previous egg plus 1 min).

Each listed claim's criterion, as applied above, is rerun with the search periods of the three
alternate definitions of the fig. S2 legend (helper `search_windows`).

In [ ]:
def outcomes_under(mode, reading="visit", first_left=5):
    o = {}
    o["fig3ab-2mm-tunnel-faster"] = "reproduced" if eval_3ab(mode, reading)[0] else "not_reproduced"
    o["fig3c-bounce-faster"] = eval_3c(mode, reading)[0]
    o["fig4b-faster-middle-preferred"] = "reproduced" if eval_4b(mode)[1][("(a) 5-10", reading, "(i)")] else "not_reproduced"
    o["fig5ab-strains-release-faster"] = "reproduced" if eval_5ab(mode, reading, first_left)[0] else "not_reproduced"
    o["fig5c-suzukii-no-preference"] = "reproduced" if eval_5c(mode, reading)[0] else "not_reproduced"
    ok7, d7 = eval_7cf(mode, reading, first_left)
    o["fig7cf-kir-merge-earlier"] = "reproduced" if ok7 else ("partially_reproduced" if d7["monotone"] else "not_reproduced")
    return o


S2 = pd.DataFrame({m: outcomes_under(m) for m in ("working", "alt1", "alt2", "alt3")})
print(S2.to_string())
details = []
for m in ("working", "alt1", "alt2", "alt3"):
    a = eval_3ab(m)[1]; c = eval_3c(m)[1]; t5 = eval_5ab(m)[1]; d7 = eval_7cf(m)[1]
    details.append({"definition": m, "3ab 2mm/4mm 5-10 s": f"{a[0]:.2f}/{a[1]:.2f} p={a[2]:.2g}",
                    "3c bounce/4mm/2mm": f"{c[0][0]:.2f}/{c[0][1]:.2f}/{c[1][1]:.2f} p={c[0][2]:.2g},{c[1][2]:.2g}",
                    "5ab merge TUT, DL, CS": t5, "5c bins": eval_5c(m)[1], "7cf merges": d7["merge"],
                    "7cf 31C sig bins": d7["31C bins past 5 s at p<0.05"]})
print(pd.DataFrame(details).to_string(index=False))
repro = [c for c in S2.index if S2.loc[c, "working"] == "reproduced"]
keeps = {c: int(sum(S2.loc[c, m] == S2.loc[c, "working"] for m in ("alt1", "alt2", "alt3"))) for c in repro}
print("\nClaims reproduced under the working definition, and how many of the three alternates keep the outcome:", keeps)


def figs2_verdict(reading="visit", first_left=5):
    s2 = pd.DataFrame({m: outcomes_under(m, reading, first_left) for m in ("working", "alt1", "alt2", "alt3")})
    rep = [c for c in s2.index if s2.loc[c, "working"] == "reproduced"]
    kp = {c: int(sum(s2.loc[c, m] == "reproduced" for m in ("alt1", "alt2", "alt3"))) for c in rep}
    v = "reproduced" if kp and min(kp.values()) == 3 else ("partially_reproduced" if kp and min(kp.values()) >= 2 else "not_reproduced")
    return v, kp, s2


print("\nThe criterion under each trial reading and first-bin reading of M5:")
for reading in ("visit", "search"):
    for fl in (5, 10):
        v, kp, s2 = figs2_verdict(reading, fl)
        print(f"  {reading} trials, first bin >= {fl} s: {v}; reproduced under the working definition, and alternates keeping it: {kp}")
        if (reading, fl) != ("visit", 5):
            print("    " + s2.to_string().replace("\n", "\n    "))

**Result: not reproduced.** With visit trials and the 5 to 10 s bin first (the readings adopted for
fig5ab and fig7cf), three of the listed claims are reproduced under the working definition.
fig3ab-2mm-tunnel-faster and fig4b-faster-middle-preferred keep that outcome under all three
alternate definitions; fig5ab-strains-release-faster keeps it only under alternate 1 (TUT merges at
90 s under alternate 2, and CS merges at 1200 s under alternate 3). The criterion asks that each
reproduced claim keep its outcome under all three (or two, for a partial outcome). With the 10 to
15 s bin first, fig7cf is reproduced too and keeps it under all three, but fig5ab still fails. With
search trials fig5ab is not reproduced under the working definition, so it drops out of the test,
and the remaining reproduced claims (fig3ab, fig4b, and fig7cf with the 10 to 15 s bin first) keep
their outcome under all three alternates: the criterion would hold. The outcome rests on the visit
reading. Among the others, fig3c-bounce-faster is reproduced under alternates 1 and 3 with visit
trials, and fig5c-suzukii-no-preference under alternates 1 and 2.

## figs3-long-searches

> Flies still lay most eggs on low sucrose even after long searches with many transitions (3979 eggs
> from 95 flies in 0/200, 0/500 and 200/500 mM chambers).

In [ ]:
rows = []
for g, low in [("0-200-CS", 0), ("0-500-CS", 0), ("500-200-CS", 200)]:
    for f in G[g]:
        for a, e in zip(f["starts"], f["eggs"]):
            seg = f["sucrose"][a:e + 1]
            rows.append({"chamber": g, "fly": f["path"], "dur": (e - a) * E.DT, "trans": int((np.diff(seg) != 0).sum()), "low": f["sucrose"][e] == low})
s3 = pd.DataFrame(rows)
print(f"Eggs {len(s3)}, flies {sum(len(G[g]) for g in ['0-200-CS', '0-500-CS', '500-200-CS'])} (flies with eggs {s3.fly.nunique()})")
dur_edges = [0, 30, 60, 90, 120, 240, 360, 1500, np.inf]
s3["dur bin"] = pd.cut(s3.dur, dur_edges, right=False)
s3["trans bin"] = pd.cut(s3.trans, [0, 1, 2, 3, 4, 5, 10, 25, np.inf], right=False, labels=["0", "1", "2", "3", "4", "5-9", "10-24", "25+"])
for col in ("dur bin", "trans bin"):
    t = s3.groupby(col, observed=False).low.agg(["size", "sum"])
    t["fraction low"] = (t["sum"] / t["size"]).round(3)
    t["95% CI"] = [f"{E.clopper(k, n)[0]:.3f} to {E.clopper(k, n)[1]:.3f}" if n else "" for k, n in zip(t["sum"], t["size"])]
    print(t.to_string())
lng = s3[s3.dur >= 360]
print(f"Longest bins pooled, searches >= 360 s: {lng.low.sum()}/{len(lng)} = {lng.low.mean():.3f}")
print("Search duration range (s):", s3.dur.min(), "to", s3.dur.max(), "; transitions range:", s3.trans.min(), "to", s3.trans.max())

**Result: reproduced.** The 3978 eggs of 95 flies (89 with eggs) are within tolerance of 3979 and
95. The fraction of eggs on the lower sucrose option is 0.949 (95% CI 0.903 to 0.978) for searches
of 360 to 1500 s, the longest bin (no search lasts 1500 s or more), and 0.957 (0.892 to 0.988) for
searches with 25 or more transitions; it is above 0.89 in every bin.

## figs4-models-fail

> Models that map sucrose history to egg-laying rate without a comparison to an internal
> expectation (a history kernel, or an integral controller with a set point) fail to reproduce key
> features of the rate functions.

The claims file's reading: a zero-area kernel is fit by least squares to the 0/200 mM data, and
the test is whether it overestimates the rate on 0 mM after leaving 500 mM in the 0/500 chamber
(Methods, Mathematical models). The kernel is piecewise constant on 13 log-spaced lag boxes up to
34 min (helper `KERNEL_BOXES`) and fit to the 0/1 egg series over search frames. Model 2 has
constants the paper does not give and is not fit.

In [ ]:
c0, beta, nfit = E.fit_kernel(G["0-200-CS"])
print(f"Fit on {nfit} search frames of the 0/200 chamber: intercept {c0 * 120:.3f} eggs/min; kernel per box (eggs/min per 200 mM):")
print(pd.DataFrame({"lags (s)": [f"{lo * E.DT:g}-{hi * E.DT:g}" for lo, hi in E.KERNEL_BOXES], "beta x 120": np.round(beta * 120, 4)}).T.to_string(header=False))
print("Kernel area (should be 0):", round(float(sum(b * (hi - lo) for b, (lo, hi) in zip(beta, E.KERNEL_BOXES))), 10))
for g in ("0-200-CS", "0-500-CS"):
    for f in G[g]:
        f["pred"] = E.predict_kernel(f, c0, beta)
s4 = {}
for g, hi in (("0-200-CS", 200), ("0-500-CS", 500)):
    for cur, prev in ((0, hi), (hi, 0)):
        fl = [dict(label=f["sucrose"], eggs=f["eggs"], starts=f["starts"], n=f["n"], pred=f["pred"]) for f in G[g]]
        r = E.rate_events(fl, cur, prev, pred_key="pred")
        lo_, hi_ = E.rate_ci(r)
        s4[(g, cur, prev)] = pd.DataFrame({"bin": BINS, "observed": np.round(r["rate"], 3), "CI low": np.round(lo_, 3), "CI high": np.round(hi_, 3),
                                           "model": np.round(r["pred_sum"] / r["frames"] * 120, 3)})
for k, t in s4.items():
    print(f"\n{k[0]}: rate on {k[1]} mM since leaving {k[2]} mM")
    print(t.to_string(index=False))
t = s4[("0-500-CS", 0, 500)]
idx = bins_in(5, 60)
over = int((t.model.values[idx] > t["CI high"].values[idx]).sum())
fit_t = s4[("0-200-CS", 0, 200)]
inside = int(((fit_t.model.values[idx] >= fit_t["CI low"].values[idx]) & (fit_t.model.values[idx] <= fit_t["CI high"].values[idx])).sum())
print(f"\n0/500, 0 mM since 500 mM, bins 5-60 s: model above the 90% CI in {over} of {len(idx)}; mean model {t.model.values[idx].mean():.3f} against observed {t.observed.values[idx].mean():.3f}")
print(f"0/200 fit, 0 mM since 200 mM, bins 5-60 s: model inside the 90% CI in {inside} of {len(idx)}")
pred_min = min(float(np.min(f["pred"])) for g in ("0-200-CS", "0-500-CS") for f in G[g]) * 120
print(f"Lowest predicted rate of the least squares fit: {pred_min:.2f} eggs/min (a negative rate)")

# Variant: a fit that cannot predict negative rates (log link, Poisson likelihood), same kernel boxes and zero area.
c0L, betaL, nL, convL = E.fit_kernel_loglink(G["0-200-CS"])
print(f"\nLog-link fit on {nL} frames, converged: {convL}; kernel area {float(sum(b * (hi - lo) for b, (lo, hi) in zip(betaL, E.KERNEL_BOXES))):.2g}")
for g in ("0-200-CS", "0-500-CS"):
    for f in G[g]:
        f["predL"] = E.predict_kernel(f, c0L, betaL, link="log")
s4L = {}
for g, hi in (("0-200-CS", 200), ("0-500-CS", 500)):
    for cur, prev in ((0, hi), (hi, 0)):
        fl = [dict(label=f["sucrose"], eggs=f["eggs"], starts=f["starts"], n=f["n"], pred=f["predL"]) for f in G[g]]
        r = E.rate_events(fl, cur, prev, pred_key="pred")
        lo_, hi_ = E.rate_ci(r)
        s4L[(g, cur, prev)] = pd.DataFrame({"bin": BINS, "observed": np.round(r["rate"], 3), "CI low": np.round(lo_, 3), "CI high": np.round(hi_, 3),
                                            "model": np.round(r["pred_sum"] / r["frames"] * 120, 3)})
for k, t in s4L.items():
    t_ = t.iloc[idx]
    print(f"{k[0]}, {k[1]} mM since leaving {k[2]} mM, bins 5-60 s: model inside the 90% CI in "
          f"{int(((t_.model >= t_['CI low']) & (t_.model <= t_['CI high'])).sum())} of {len(idx)}, above it in {int((t_.model > t_['CI high']).sum())}; "
          f"mean model {t_.model.mean():.3f} against observed {t_.observed.mean():.3f}")
print("\n0/500, 0 mM since 500 mM, log-link model:")
print(s4L[("0-500-CS", 0, 500)].to_string(index=False))

**Result: reproduced.** The zero-area kernel fit to the 0/200 mM chamber by least squares follows
the 0 mM curve roughly (inside its 90% CI in 3 of 7 bins from 5 to 60 s) and, applied to the 0/500
mM chamber, predicts a higher rate on 0 mM after leaving 500 mM than observed in all 7 bins from 5
to 60 s (mean 1.82 against 1.23 eggs/min, every bin above the 90% CI), the failure the Methods
describe. This fit predicts negative rates (as low as -2.69 eggs/min). A fit that cannot, with a log
link and Poisson likelihood on the same kernel, follows the 0/200 curves somewhat better (inside the
90% CI in 4 of 7 bins on each curve; mean 1.27 against 1.21 eggs/min on 0 mM) and overestimates the
0/500 rate on 0 mM after 500 mM still more (7 of 7 bins above the CI, mean 8.08 against 1.23). Both
fits put sucrose into the model linearly in concentration, as model 1 does, so a 500 mM history
weighs 2.5 times a 200 mM one; that is the paper's account of the failure. The outcome rests on the
kernel basis and fit taken here. Model 2 was not fit: the paper gives neither its equations nor its
constants k1 and k2, and the criterion tests model 1 only.

## Blocked claims

These claims need experiments or labels the dandiset does not hold (Data check, above):

| Claim | Needs | Reason |
|---|---|---|
| fig1d-no-substrate-marking | movable-boundary chambers (Fig. 1D, E) | data missing |
| fig1f-tarsi-proboscis | flies with glued tarsi or proboscis | data missing |
| fig6b-softer-preferred | sucrose on the softer side only (bars 2 and 3), one of the two equal-sucrose bars, and 500 mM on 0.8% against 200 mM on 1.4% (bar 6) | data missing |
| fig7a-screen-ddc-hits | the 115 Gal4 lines of the screen, Table S2 | data missing |
| fig7b-ddc-kir-sense-sucrose | latency-to-feeding videos or scores | data missing |
| fig7l-dopamine-gal80-rescue | TH-Gal80 and R58E02-Gal80 flies | data missing |
| fig7mn-brain-neurons | TSH-Gal80 and OTD-FLP flies | data missing |
| fig8a-d1-receptor-mutants | nine of the 13 genotypes, among them Dop1R1 f02676 | data missing |
| fig8b-mutants-sense-sucrose | latency-to-feeding videos or scores | data missing |
| fig8j-200-500-defect | 200 against 500 mM chambers for mutants and DDC>Kir | data missing |
| fig8k-agarose-intact | 0.8 against 1.4% chambers for mutants and DDC>Kir | data missing |
| dop1r1-84714-no-defect | the BDSC 84714 Dop1R1 mutant | data missing |

For fig8a, the three receptor mutants that are in the dandiset are shown below, not as an outcome; for fig6b, the
bars it holds (the 0.8/1.4% chamber, and one with 200 mM on both sides, which may be bar 4 or 5).

In [ ]:
for k, g in MUT.items():
    (fr, lo, hi), kk, n = egg_frac(g)
    print(f"{k}: {kk}/{n} eggs on 0 mM, fraction {fr:.3f} (95% CI {lo:.3f} to {hi:.3f}); CS {egg_frac('0-200-CS')[0][0]:.3f}")
print("\nfig6b, the conditions of Fig. 6B that the dandiset holds (not an outcome):")
for g in ["0p8--1p4--CS", "0p8-200-1p4-200-CS"]:
    k = sum(int((f["agarose"][f["eggs"]] == 8).sum()) for f in G[g]); n = sum(len(f["eggs"]) for f in G[g])
    fr, lo, hi = E.frac_ci(k, n)
    print(f"  {g}: {k}/{n} eggs on 0.8%, fraction {fr:.3f} (95% CI {lo:.3f} to {hi:.3f}); mean eggs per fly {n / len(G[g]):.1f} ({len(G[g])} flies)")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| leave-higher-stay-lower | reproduced |
| fig1d-no-substrate-marking | blocked: data missing |
| fig1f-tarsi-proboscis | blocked: data missing |
| fig2bc-larger-chambers-two-options | reproduced |
| fig2d-three-choice-inhibit-200 | reproduced |
| fig2e-200-recovers-with-time | reproduced |
| fig2efg-inhibit-200-after-0 | reproduced |
| fig2-earlier-merge-three-choice | reproduced |
| fig3ab-2mm-tunnel-faster | reproduced |
| fig3c-bounce-faster | partially reproduced |
| fig3d-dwell-before-fast-eggs | partially reproduced |
| fig3e-rate-after-long-absence | partially reproduced (rests on an assumption) |
| fig3f-motor-program-not-faster | not reproduced |
| fig4a-shorter-middle-transits | partially reproduced |
| fig4b-faster-middle-preferred | reproduced |
| fig4d-anomalous-eggs-alternative | reproduced |
| fig4e-inter-egg-not-longer | reproduced |
| fig4f-motor-program-not-accelerated | reproduced |
| fig5ab-strains-release-faster | reproduced (rests on an assumption) |
| fig5c-suzukii-no-preference | not reproduced |
| fig5de-suzukii-more-on-200 | reproduced |
| fig5f-egg-numbers-differ | reproduced |
| fig5gh-search-and-pause | reproduced |
| fig6b-softer-preferred | blocked: data missing |
| fig6cd-hardness-fast-merge | reproduced |
| fig6ef-pause-longer-hard | reproduced |
| fig6g-more-time-softer | reproduced |
| fig7a-screen-ddc-hits | blocked: data missing |
| fig7b-ddc-kir-sense-sucrose | blocked: data missing |
| fig7cf-kir-merge-earlier | partially reproduced (rests on an assumption) |
| fig7df-overall-rate-drops | partially reproduced |
| fig7gh-choice-falls | reproduced |
| fig7ik-not-egg-number-or-motor | not reproduced |
| fig7l-dopamine-gal80-rescue | blocked: data missing |
| fig7mn-brain-neurons | blocked: data missing |
| fig8a-d1-receptor-mutants | blocked: data missing |
| fig8b-mutants-sense-sucrose | blocked: data missing |
| fig8ce-mutants-flat-rates | reproduced |
| fig8fg-time-and-eggs | partially reproduced |
| fig8j-200-500-defect | blocked: data missing |
| fig8k-agarose-intact | blocked: data missing |
| dop1r1-84714-no-defect | blocked: data missing |
| figs2-search-definition-robust | not reproduced (rests on an assumption) |
| figs3-long-searches | reproduced |
| figs4-models-fail | reproduced (rests on an assumption) |

How the analysis differs from the paper's. The paper's code is not public, so every measure is
rebuilt from the Methods and the claims file's shared methods. The search start scans back from 9
s before the egg (an assumption that gives the paper's 1205 searches on higher sucrose exactly).
The rank sum test is tie-corrected. Where the claims file names readings of the bin test (one
trial per visit or per search), both are shown, and the visit reading is adopted where they differ
(fig5ab, figs2), as the Methods treat each substrate transition as an independent trial; the merge
point takes the 5 to 10 s bin as the first bin past 5 s, an assumption on which fig7cf depends.
Merge points can fall on bins with almost no eggs just after a transition (fig2-earlier-merge).
In fig3e the "no rising trend" below 240 s is judged with the step at 150 to 240 s and the bins from
5 s; over all bins it would hold. Working search windows overlap when a 30-s extension reaches into
the previous search; the overlap is 1.6% of the frame counts. Egg-aligned averages leave out eggs
within 2 min of either end of a recording, a standard choice that gives the legends' egg counts
in most groups. The fig. S4 kernel is a piecewise-constant zero-area kernel fit by least squares;
the paper's fit (ref. 82) is not given; a log-link fit gives the same result. Pause times by the
Methods' definition are short (medians 0.5 to 2.5 s) and whole numbers of 0.5-s frames, so the
claims file's 10% and 20% bounds on their medians (fig3f, fig7ik) and the 10% tolerance on the 4.5-s
transit median (fig4a) come down to a single frame.

What was found about the dataset. The files agree with the paper wherever the paper gives a
count: egg counts per group are within a few eggs of the legends, the ref. 16 chambers hold 95
flies and 3978 eggs (paper 3979), and the egg annotations share the position clock. No recording
is deposited twice. Several experiments of the paper are not in the dandiset, which describes
itself as holding all the paper's tracking data. The agarose chambers record the agarose under the
fly in a series the other groups do not have, and were loaded both ways round. The two Dop1R2 groups are named only by the paper the mutant came from; every
file carries the same placeholder session start time; 48 intervals between eggs are under 10 s,
and 10 eggs are annotated in the last 19 s of two Dop1R2-mutant-1 recordings while the flies walk;
and the dandiset has never been published, so it has no fixed version or DOI.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")
import resource
print(f"Peak memory: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")